# Trend Following / Time Series Momentum — Guía completa del proyecto
### Ingeniería Financiera (F414) · Universidad de San Andrés · TP final (2026)

Este es el **único notebook** del proyecto: la guía de punta a punta para **entender, presentar y usar** el
trabajo. Recorre el sistema **en orden de construcción** —de la intuición económica y los papers, a cada módulo
de código, su matemática y cómo todo se conecta— y, en cada paso, muestra **cómo eso se ve en el dashboard** y
**cómo se usa**.

La regla de oro: **el dashboard (`src/dashboard/app.py`) es solo una capa de presentación**. Todos los números
salen de `src/`. Acá llamamos *exactamente las mismas funciones del backend*, así que lo que ves en este
notebook es idéntico a lo que muestra la terminal. Cada celda de código reproduce un número o gráfico con el
motor real.

> Con este notebook a mano alcanza para reconstruir y defender el proyecto entero: qué hace cada cosa y por qué,
> de dónde salen los datos, cómo se backtestea, qué tan robusto es y cómo se opera en vivo.

**Papers que fundamentan la estrategia**

| Rol | Paper | Qué aporta al proyecto |
|---|---|---|
| Principal `[L]` | Moskowitz, Ooi & Pedersen (2012) — *Time Series Momentum*, JFE | señal de signo, normalización por vol ex-ante, target-vol por activo, rebalanceo mensual, universo multi-asset |
| Complementario `[H]` | Baz et al. (2015) — *Dissecting Investment Strategies* | señal **continua** (`strength`) normalizada por dispersión; multi-horizonte |
| Adicional | Daniel & Moskowitz (2016) — *Momentum Crashes*, JFE | **filtro de crash** (recortar exposición en regímenes de pánico) |
| Adicional | Moreira & Muir (2017) — *Volatility-Managed Portfolios*, JF | **vol-scaling del portafolio** (escalar la cartera entera al riesgo objetivo) |
| Soporte | Hurst-Ooi-Pedersen (2017); Barroso & Santa-Clara (2015); Maillard-Roncalli-Teïletche (2010) | 100+ años de evidencia; vol-scaling reduce crashes; risk parity vs 1/N |

---
### Cómo correr este notebook
Usar el kernel del env **`finance`** (Python ≥ 3.13, pandas ≥ 2.2), desde la **raíz del proyecto**
(`TP_Trading_Strategy/`):
```bash
conda activate finance
jupyter lab TP_walkthrough.ipynb
```
> El `python`/anaconda base (3.11, pandas 2.1) **no** corre todo el backend: `pandas < 2.2` no entiende la
> frecuencia `"ME"` (month-end) que usa `src/data/riskfree.py`. Con el env `finance` corre completo.

---
## Índice
**Concepto y arquitectura**
1. [Intuición económica y papers](#s1)
2. [Arquitectura del sistema](#s2)

**El sistema, módulo por módulo (en orden de construcción)**
3. [Capa de datos — `src/data/` (universo · loader · riskfree)](#s3)
4. [Estrategia TSMOM — `src/strategy/` (volatilidad · señal · pesos)](#s4)
5. [Backtesting — `src/backtest/` (costos · métricas · motor)](#s5)
6. [Robustez — `src/robustness/` (sensibilidad · stress · out-of-sample)](#s6)
7. [Broker IBKR y trading en vivo — `src/broker/`](#s7)

**Cómo se usa**
8. [CLI — `main.py`](#s8)
9. [Dashboard — `src/dashboard/app.py` (6 páginas)](#s9)
10. [Configuración central — `config.yaml`](#s10)
11. [Demo end-to-end](#s11)
12. [Resumen de decisiones de diseño + cómo correr todo](#s12)


In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  SETUP — cargamos el backend real UNA sola vez (mismas funciones del dashboard)
# ════════════════════════════════════════════════════════════════════════
import sys; sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
pd.set_option('display.precision', 4)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

# --- backend real del proyecto (las mismas funciones que usa el dashboard) ---
from src.data.loader import load_returns
from src.data.universe import ETF_UNIVERSE, ASSET_CLASSES
from src.strategy.signals import compute_tsmom_signal, compute_signal_strength
from src.strategy.volatility import compute_ex_ante_vol
from src.strategy.portfolio import (
    compute_vol_scaled_weights, compute_classbalanced_weights, build_portfolio,
)
from src.backtest.engine import BacktestEngine
from src.backtest.metrics import (
    sharpe_ratio, annualized_return, annualized_volatility, max_drawdown, calmar_ratio,
)
from src.robustness.sensitivity import optimize_sharpe

config  = yaml.safe_load(open('config.yaml'))
returns = load_returns(config)   # retornos MENSUALES de los ETF del universo activo

# IMPORTANTE — reproducimos el estado "al abrir el dashboard".
# La terminal, al cargar, corre optimize_sharpe (grid lookback 3-24m × target-vol 5-40%) para la
# señal/ponderación elegidas y FIJA los sliders al óptimo in-sample. Todos los Sharpes que mostramos
# en este notebook usan ESE óptimo (no el 12m/10% que config.yaml guarda como base).
opt, _ = optimize_sharpe(returns, config)          # combo por defecto: strength | pooled
config['strategy']['lookback_months']   = opt['lookback']
config['strategy']['target_volatility'] = opt['target_vol']

print(f"Universo activo : {returns.shape[1]} ETFs")
print(f"Ventana         : {returns.index[0].date()}  ->  {returns.index[-1].date()}  ({len(returns)} meses)")
print(f"Óptimo (strength|pooled): Sharpe {opt['sharpe']:.3f}  @ lookback {opt['lookback']}m · target vol {opt['target_vol']*100:.0f}%")
config['strategy']


---
<a id='s1'></a>
## 1. Intuición Económica y Base Bibliográfica

### ¿Qué es Time Series Momentum (TSMOM)?

La idea es simple: **los activos que subieron en los últimos 12 meses tienden a seguir subiendo el mes siguiente**, y los que bajaron tienden a seguir bajando. A diferencia del momentum transversal (que compara activos entre sí), el TSMOM evalúa cada activo contra su propio historial.

**Señal:**
$$s_{i,t} = \text{sign}\left(\sum_{k=1}^{12} r_{i,t-k}\right) = \begin{cases} +1 & \text{si el activo subió → COMPRAR} \\ -1 & \text{si el activo bajó → VENDER (short)} \end{cases}$$

### ¿Por qué funciona? (3 teorías)

| Teoría | Mecanismo |
|--------|-----------|
| **Underreaction** | Los inversores reaccionan lento a nueva información → el precio ajusta gradualmente |
| **Herding** | Los managers compran lo que subió por presión de clientes → auto-cumplimiento |
| **Delegated management** | Los hedge funds tienen mandatos que los obligan a seguir tendencias |

### Base Bibliográfica del Proyecto

| Rol | Paper | Contribución al proyecto |
|-----|-------|-------------------------|
| **[L] Principal** | Moskowitz, Ooi & Pedersen (2012) — *Time Series Momentum*, JFE | Diseño de señal, vol-scaling, universo multi-asset |
| **[H] Complementario** | Baz et al. (2015) — *Dissecting Investment Strategies* | Señal continua normalizada (`signal_strength`), multi-timescale |
| **Adicional 1** | Hurst, Ooi & Pedersen (2017) — *A Century of Evidence on Trend-Following*, JPM | Valida que TSMOM funciona en 100+ años y múltiples asset classes |
| **Adicional 2** | Barroso & Santa-Clara (2015) — *Momentum Has Its Moments*, JFE | Escalar por volatilidad reduce crashes — justifica nuestro EWMA vol-scaling |

---
<a id='s2'></a>
## 2. Arquitectura del sistema

El proyecto se divide en **capas independientes** con comunicación unidireccional: los datos fluyen hacia
arriba (de `data/` a la estrategia, al backtest, a la robustez y al dashboard) y las órdenes bajan hacia IBKR.

```
┌────────────────────────────────────────────────────────────────────────────┐
│                       main.py  (CLI — Click, 15 comandos)                    │
│  update-data · data-status · backtest · robustness · dashboard               │
│  test-connection · validate-ibkr · monitor · execute                         │
│  start-strategy · halt · resume · tick · close-all · strategy-status         │
└──────┬──────────────────────────────────────────────────────┬───────────────┘
       │                                                       │
  ┌────┴──────────────────┐                          ┌─────────┴────────────────┐
  │  src/data/            │  205→133 ETFs            │  src/broker/  (IBKR)      │
  │  ├─ universe.py       │  qué ETFs entran         │  ├─ ibkr.py    (conexión) │
  │  ├─ loader.py         │  precios → retornos      │  ├─ session.py (client ids)│
  │  └─ riskfree.py       │  T-bill 1m               │  ├─ orders.py  (ejecuta)  │
  └────┬──────────────────┘                          │  ├─ runner.py  (tick)     │
       │                                              │  ├─ state.py   (cerebro)  │
  ┌────▼──────────────────┐                          │  ├─ monitor.py (P&L vivo) │
  │  src/strategy/        │  σ̂ ex-ante              │  └─ market_hours.py       │
  │  ├─ volatility.py     │  s_{i,t}  (señal)        └───────────────────────────┘
  │  ├─ signals.py        │  w_{i,t}  (pesos)
  │  └─ portfolio.py      │  + crash filter, class_balanced
  └────┬──────────────────┘
       │
  ┌────▼──────────────────┐  el MISMO motor que opera en vivo
  │  src/backtest/        │  + vol-scaling de portafolio (Moreira-Muir)
  │  ├─ engine.py         │
  │  ├─ costs.py          │
  │  └─ metrics.py        │
  └────┬──────────────────┘
       │
  ┌────┴───────────────┐   ┌────────────────────────────────────┐
  │  src/robustness/   │   │  src/dashboard/app.py  (Streamlit)  │
  │  ├─ sensitivity.py │   │  6 páginas:                         │
  │  ├─ stress_test.py │   │  01 Overview · 02 Backtest          │
  │  └─ out_of_sample  │   │  03 Robustness · 04 Live Portfolio  │
  └────────────────────┘   │  05 Trade Log · 06 Rebalances       │
                           └────────────────────────────────────┘
```

**Principios de diseño**
- **Una responsabilidad por módulo**: `strategy/` solo produce señales y pesos; `backtest/` solo simula. Ningún módulo conoce la implementación de otro.
- **`config.yaml` como única fuente de verdad**: todos los parámetros (lookback, target-vol, costos, fechas, broker) viven ahí.
- **Sin look-ahead bias**: cada cálculo usa `.shift(1)` donde corresponde → en el mes `t` solo se usa información hasta `t-1`.
- **El backtest *es* lo que se opera**: `BacktestEngine` y la ejecución en vivo en IBKR usan **exactamente los mismos pesos** (`build_scaled_weights`).
- **El dashboard es presentación pura**: lee de `src/`, sin lógica propia. Este notebook llama esas mismas funciones.


---
<a id='s3'></a>
## 3. Capa de datos — `src/data/`

Tres archivos: **`universe.py`** (qué ETFs entran), **`loader.py`** (precios → retornos mensuales) y **`riskfree.py`** (la T-bill a 1 mes; se usa en §5 para medir el exceso). Empezamos por la decisión que condiciona todo lo demás — **el universo**.

> **En el dashboard:** esta curación es la página **00 · Selección del universo** (y el KPI "Universo" del Overview).

### 3.1 `universe.py` — De 205 ETFs crudos a 133 mercados distintos

> **El planteo.** Antes de cualquier señal hay que decidir **qué ETFs entran y cuáles no**. Esa decisión
> condiciona todo lo que sigue (la señal, la ponderación, el riesgo del portafolio), así que conviene tenerla
> documentada y poder justificarla. La idea: un universo **amplio** (muchas clases y mercados, para que el
> TSMOM diversifique de verdad) pero **operable y sin redundancias** (que represente mercados *distintos*, no
> el mismo índice diez veces).

Partimos de un universo **crudo de 205 ETFs** y lo curamos hasta **133 mercados distintos**, con tres reglas.
La fuente de verdad es `src/data/universe.py` (la celda de abajo reproduce los números).

### Regla 1 — Solo US-listed en USD
Todos los ETFs cotizan en EE.UU. y en dólares. **No** entran ETFs extranjeros (`.MI` de Milán, `.TO` de
Toronto, etc.): sus retornos mezclan el movimiento del activo con el del tipo de cambio (EUR/CAD vs USD), lo
que **contamina** una estrategia 100% en USD. Esos quedan registrados en `EXCLUDED_TICKERS` con el motivo.

### Regla 2 — Cada ETF desde su *inception*, sin lookahead
Se incluye un ETF **sin importar cuándo arrancó**: se usa desde que existe, y los meses previos quedan `NaN`
(el pipeline los trata como "activo inactivo", peso 0). Así evitamos *survivorship bias* (no descartamos lo
que arrancó tarde) y *lookahead bias* (nunca usamos un activo antes de que existiera). El único requisito es
tener historia mínima para formar señal (`data.min_history_months` en `config.yaml`).

### Regla 3 — Curación por liquidez (corr ≥ 0.95) → 1 instrumento por *mercado*
Acá está el punto fino. Siguiendo a **Moskowitz et al. (2012)** —que usa **un instrumento líquido por
mercado**— agrupamos los ETFs por **correlación de retornos mensuales** (complete-linkage, **corr ≥ 0.95** =
"el mismo mercado") y nos quedamos con **el más líquido** de cada grupo (mayor dollar-volume). Los **72**
redundantes van a `REDUNDANT_TICKERS`, cada uno con el ticker que lo reemplaza. Así pasamos de **205 → 133**.

> ⚠️ **"Mismo mercado" ≠ "mismo sector/clase".** El cluster junta *clones del mismo índice*, no clases de
> activo. Por ejemplo, `SPY` absorbe a `IVV, VOO, VTI, ITOT, IWB, OEF, QUAL` (todos S&P 500 / total-US, corr
> > 0.95); `GLD` absorbe `IAU, SGOL`; `EEM` absorbe `IEMG, VWO`. Lo que se saca son **duplicados** del mismo
> mercado, **no** diversidad entre clases.

### Consecuencia (clave para `pooled` vs `class_balanced`, sección B)
La curación **no** dejó "un ETF por clase": el universo activo sigue **muy sesgado a equity** (~89 de 133
nombres ≈ 67%; ver celda). Por eso las dos ponderaciones de la sección B son **genuinamente distintas**:
`pooled` (1/N global) deja que equity —con muchos más nombres— domine el riesgo, mientras que
`class_balanced` reparte el presupuesto parejo entre las 4 clases. **Solo** serían idénticas si hubiéramos
colapsado a *un único ETF por clase* (4 en total), y eso **no** se hizo: curamos por *redundancia de mercado*,
no por sector.

> Scripts de soporte: `scripts/universe_redundancy.py` arma los clusters de correlación;
> `scripts/compare_universe_backtest.py` compara el backtest con 205 vs 133; `python main.py data-status`
> chequea que `universe.py` y `Data/etf_prices.parquet` sigan alineados. Para volver a los 205, basta vaciar
> `REDUNDANT_TICKERS` (los datos siguen en el parquet).

In [ ]:
# --- Selección del universo: de 205 ETFs crudos a 133 mercados distintos ---
from collections import Counter
from src.data.universe import _ALL_ETFS, ETF_UNIVERSE, REDUNDANT_TICKERS, EXCLUDED_TICKERS

print(f"Universo CRUDO   : {len(_ALL_ETFS)} ETFs US-listed en USD")
print(f"Redundantes      : -{len(REDUNDANT_TICKERS)}  (corr >= 0.95, se queda el más líquido de cada grupo)")
print(f"Excluidos a mano : -{len(EXCLUDED_TICKERS)}  (.MI/.TO en EUR/CAD: ruido de FX)")
print(f"Universo ACTIVO  : {len(ETF_UNIVERSE)} mercados distintos\n")

crudo  = Counter(_ALL_ETFS.values())
activo = Counter(ETF_UNIVERSE.values())
tabla = pd.DataFrame({'crudo (205)': crudo, 'activo (133)': activo}).reindex(
    ['equity', 'bond', 'commodity', 'currency'])
tabla['% nombres (activo)'] = (tabla['activo (133)'] / tabla['activo (133)'].sum() * 100).round(1)
print("Composición por clase de activo:")
print(tabla.to_string())

# Ejemplo de UN cluster de liquidez: 'mismo mercado' (corr >= 0.95), NO 'mismo sector'
spy_cluster = sorted(t for t, v in REDUNDANT_TICKERS.items() if 'SPY' in v)
print(f"\nEjemplo — el cluster del S&P 500 colapsa en SPY (el más líquido).")
print(f"   Se descartan por redundantes: {spy_cluster}")
print("   (todos son S&P 500 / total-US: el MISMO mercado duplicado, no clases distintas)")

### 3.2 `loader.py` — De precios a retornos mensuales

El loader transforma el parquet de **precios diarios** en la matriz de **retornos mensuales** que consume toda la
estrategia, en tres funciones encadenadas:

```
Data/etf_prices.parquet
   │  load_etf_prices(path)
   ▼  precios diarios [días × tickers]   (índice de fechas ordenado)
   │  compute_monthly_returns(prices)
   ▼  resample("ME").last().ffill() → pct_change()   → retornos mensuales [meses × tickers]
   │  load_returns(config)
   ▼  filtro de fechas (start … último mes COMPLETO) + piso de historia  → DataFrame listo
```

**Decisiones de diseño (todas en `src/data/loader.py`):**
- **Fin de mes (`resample("ME").last()`)** — tomamos el **último** precio del mes (no el promedio): estándar académico y sin forward-looking. `"ME"` existe en pandas ≥ 2.2; el loader cae a `"M"` en versiones viejas.
- **Cada ETF desde su *inception*** — los meses previos a que el ETF existiera quedan **`NaN`** y el pipeline los trata como "activo inactivo" (peso 0). **No** se rellena pre-inception: hacerlo sería *lookahead*. Esto evita además *survivorship bias* (no descartamos lo que arrancó tarde).
- **Piso de historia (`min_history_months = 24`)** — un ETF entra al backtest solo si tiene **≥ 24 retornos mensuales**. Descarta ETFs tan nuevos que nunca llegan a formar una señal (≈ 2× el lookback máximo).
- **Ventana abierta** — si `backtest.end_date` está vacío, `load_returns` usa hasta el **último mes calendario completo** (`last_complete_month_end()`): nunca entra un mes parcial y la ventana **avanza sola** a medida que yfinance actualiza el parquet.
- **Actualización incremental (`update_prices`)** — descarga vía yfinance **solo desde la última fecha del parquet**; si ya llega a hoy, no toca la red. `python main.py data-status` chequea que `universe.py` y el parquet sigan alineados.


In [ ]:
from src.data.loader import load_etf_prices, load_returns

with open('config.yaml') as f:
    config = yaml.safe_load(f)

prices = load_etf_prices(config['data']['path'])
returns = load_returns(config)

print(f'Precios diarios : {prices.shape[1]} ETFs × {prices.shape[0]} días')
print(f'  Rango         : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'\nRetornos mensual: {returns.shape[1]} ETFs × {returns.shape[0]} meses')
print(f'  Rango         : {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'\nEstadísticas de retornos mensuales (pooled):')
pooled = returns.values.flatten()
pooled = pooled[~pd.isna(pooled)]
print(f'  Media   : {pooled.mean():.4f}  ({pooled.mean()*12:.2%} anualizado)')
print(f'  Std     : {pooled.std():.4f}')
print(f'  Skew    : {pd.Series(pooled).skew():.2f}')
print(f'  Kurtosis: {pd.Series(pooled).kurtosis():.2f}')

---
<a id='s4'></a>
## 4. Estrategia TSMOM — `src/strategy/`

> **En el dashboard:** todo lo de esta sección alimenta la página **01 · Overview** — la "foto del próximo rebalanceo": señal por activo, exposición por clase, NAV y KPIs.


Tres archivos que implementan la lógica del paper, en orden de dependencia:

```
volatility.py  →  signals.py  →  portfolio.py
   (σ̂_{i,t})      (s_{i,t})      (w_{i,t})
```

### 4.1 `src/strategy/volatility.py` — Volatilidad ex-ante

**¿Por qué necesitamos la volatilidad?**  
Si ponemos igual capital en USO (vol ~40%) y TLT (vol ~10%), el 95% del riesgo viene de oil.
La solución es **igualar la contribución al riesgo** de cada activo.

**Fórmula EWMA (Moskowitz Eq. 1):**
$$\hat{\sigma}_{i,t}^2 = \frac{\sum_{k=0}^{\infty} (1-\delta)\,\delta^k \, r_{i,t-1-k}^2}{\text{normalización}}$$

Implementado con `pandas.ewm(com=3)` donde `com=3 meses ≈ 60 días de trading` (parámetro del paper).

**El `.shift(1)` es crítico:**  
Sin él, $\hat{\sigma}_{i,t}$ incluiría $r_{i,t}$ (el retorno de este mes), que todavía no conocemos cuando tomamos la decisión. Eso es *look-ahead bias*.

```python
# Con look-ahead bias (INCORRECTO):
ewma_var = (returns ** 2).ewm(com=3).mean()

# Sin look-ahead bias (CORRECTO):
ewma_var = (returns ** 2).ewm(com=3).mean().shift(1)
```

In [ ]:
from src.strategy.volatility import compute_ex_ante_vol

vol = compute_ex_ante_vol(returns, com_months=3)

median_vol = vol.median().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

top15 = median_vol.head(15)
axes[0].barh(range(len(top15)), top15.values * 100, color='steelblue', alpha=0.8)
axes[0].set_yticks(range(len(top15)))
axes[0].set_yticklabels(top15.index)
axes[0].set_xlabel('Volatilidad anualizada mediana (%)')
axes[0].set_title('Top 15 ETFs más volátiles')

for ticker in ['SPY', 'TLT', 'GLD', 'USO']:
    if ticker in vol.columns:
        axes[1].plot(vol.index, vol[ticker] * 100, label=ticker, alpha=0.85)
axes[1].set_title('Volatilidad ex-ante — evolución temporal')
axes[1].set_ylabel('Vol anualizada (%)')
axes[1].legend()

plt.tight_layout(); plt.show()

print('Mediana de vol por clase de activo:')
for ac in ASSET_CLASSES:
    tc = [t for t in vol.columns if ETF_UNIVERSE.get(t) == ac]
    if tc:
        print(f'  {ac:12}: {vol[tc].median().median():.1%}')

### 4.2 `signals.py` — La señal TSMOM y sus parámetros

La intuición económica (Moskowitz et al. 2012): **los activos que vinieron subiendo tienden a seguir
subiendo, y los que vinieron cayendo a seguir cayendo**, en un horizonte de ~1 a 12 meses. Es una anomalía
documentada en decenas de mercados y clases de activo.

### Paso 0 — Volatilidad ex-ante (normaliza el riesgo de cada activo)
Antes de la señal calculamos cuánto "se mueve" cada activo, con un **EWMA** (media móvil exponencial) de los
retornos al cuadrado (Moskowitz 2012, Eq. 1):

$$\sigma_{i,t} \;=\; \sqrt{12 \cdot \mathrm{EWMA}_{\text{com}=3}\!\big(r_{i,\tau}^2\big)\big|_{\tau \le t-1}}$$

- **center-of-mass = 3 meses** (≈ 60 ruedas, el equivalente al paper diario). Es el parámetro `vol_com_months`.
- el `×12` **anualiza** la varianza mensual.
- el `shift(1)` (mirar solo hasta `t-1`) evita **look-ahead bias**: en `t` no usamos información de `t`.

### Paso 1 — La señal: dos modos

**`binary`** (Moskowitz 2012) — solo el *signo* del retorno acumulado en el lookback:

$$s^{\text{bin}}_{i,t} \;=\; \operatorname{sign}\!\left(\sum_{k=1}^{L} r_{i,\,t-k}\right) \;\in\;\{-1,\,+1\}$$

> **Cómo funciona.** Mira el retorno acumulado de los últimos $L$ meses y se queda **solo con el signo**: la
> posición es full-long o full-short, sin grados. Toda la *magnitud* la aporta después el vol-scaling
> (bloque B), no la señal — la señal solo decide **dirección**.
>
> **Crítica.** *(i) Tira la magnitud:* una tendencia apenas positiva pesa igual que una fortísima.
> *(ii) Discontinuidad en cero:* cuando el acumulado cruza 0, la posición salta de $+1$ a $-1$ de golpe, así
> que en activos "planos" / sin tendencia clara genera *flips* y turnover (= costos) sin información real.
> *(iii) Como contracara, es robusta:* al ser solo el signo, no la afectan outliers ni el ruido de estimar
> una dispersión — un parámetro menos que calibrar. Por eso rinde casi igual al `strength` (≈0.69) siendo
> más simple.

**`strength`** (Baz et al. 2015) — señal *continua*: el mismo retorno acumulado pero **normalizado** por su
propia dispersión histórica y recortado a `[-3, 3]`, así captura *dirección + magnitud* de la tendencia:

$$s^{\text{str}}_{i,t} \;=\; \operatorname{clip}\!\left(\frac{\sum_{k=1}^{L} r_{i,\,t-k}}{\operatorname{std}_{2L}\!\big(\sum r\big)},\; -3,\; 3\right)$$

> **Cómo funciona.** Toma el mismo retorno acumulado pero lo divide por su **propia dispersión histórica**
> (la `std` del acumulado sobre una ventana de $2L$ meses) y lo recorta a $[-3, 3]$. El resultado es una
> especie de *t-stat de la tendencia*: cercano a $\pm 3$ si la tendencia es fuerte **respecto de su propio
> historial**, cerca de 0 si es débil. Así pondera dirección **y** magnitud.
>
> **Crítica.** *(i) Ruido de estimación en el denominador:* la `std` necesita ~$2L$ meses de historia (NaN al
> arranque) y se vuelve inestable cuando la dispersión es chica (en el código, `std=0` → `NaN`).
> *(ii) El clip a $[-3, 3]$ es arbitrario* (heredado de Baz): frena posiciones gigantes, pero también recorta
> tendencias genuinamente extremas. *(iii) Redundancia parcial con el vol-scaling:* la señal ya normaliza por
> la dispersión del acumulado y el bloque B **vuelve** a dividir el peso por $\sigma_i$; como
> $\operatorname{std}(\sum r)\approx\sqrt{L}\,\sigma_i$, parte de la magnitud que aporta `strength` se
> "re-desinfla" después. Su aporte real sobre `binary` no es la magnitud cruda, sino *qué tan fuerte es la
> tendencia relativa a su propio historial*. Es el default porque ese matiz suma un toque (≈0.76) y fue más
> estable fuera de muestra.

> En el dashboard, el toggle **Señal** elige entre `binary` y `strength`. La **matriz de Overview** muestra
> el *signo* (color long/short) con `binary`, y la *intensidad* de la barra con `|strength| / 3`.
> Al óptimo de cada una (ver A.bis), las dos son competitivas in-sample (**`strength` ≈0.76** vs
> **`binary` ≈0.69** de Sharpe). El default es **`strength`** porque, además de un toque más alto, resultó
> **más estable fuera de muestra** en el walk-forward (sección *Robustness*).

Existió un tercer modo, **`multihorizon`** (Baz et al. 2015): promedia la señal `strength` sobre varios
lookbacks `[3, 6, 12]` para no depender de un único horizonte. Lo **probamos y lo sacamos** del dashboard
porque no mejora en este universo de ETF con rebalanceo mensual — ver la sección **A.bis** abajo.

### Parámetros de la señal (resumen)
| Parámetro (config) | Símbolo | Valor | Qué controla |
|---|---|---|---|
| `lookback_months` | $L$ | 12 base · **óptimo al abrir** (p.ej. 17m en strength\|pooled) | cuántos meses de historia mira la tendencia |
| `vol_com_months` | com | 3 | memoria del EWMA de volatilidad |
| `signal_mode` | — | `strength` | binary / strength (multihorizon: ver A.bis, fuera del dashboard) |
| `target_volatility` | $\sigma_{tgt}$ | 0.10 base · **óptimo al abrir** (p.ej. 21%) | riesgo objetivo por activo (ver bloque B) |
| `max_position_weight` | — | 0.15 | tope de peso por activo (±15%) |

> **Nota sobre los parámetros.** El `config.yaml` guarda `lookback=12` / `target_vol=10%` como *base*, pero
> el dashboard corre `optimize_sharpe` al abrir y **fija ambos al óptimo in-sample** de la señal/ponderación
> elegida. Por eso este notebook (celda de setup) también arranca en el óptimo — para que sus números
> coincidan con la terminal.

In [ ]:
# --- reproducimos la 'foto' de señales del último corte (lo que muestra Overview) ---
lookback = config['strategy']['lookback_months']

strength = compute_signal_strength(returns, lookback=lookback)   # señal continua (Baz 2015)
binary   = compute_tsmom_signal(returns,   lookback=lookback)    # solo el signo (Moskowitz 2012)

last_dir = binary.iloc[-1].dropna()      # +1 long / -1 short por activo
last_str = strength.iloc[-1]             # intensidad por activo

n_long  = int((last_dir ==  1).sum())
n_short = int((last_dir == -1).sum())
net_exp = (n_long - n_short) / max(len(last_dir), 1)

print(f"Corte {returns.index[-1].date()}  ·  lookback {lookback}m")
print(f"{n_long} LONG  /  {n_short} SHORT   ->   net exposure {net_exp:+.0%}")

In [ ]:
from src.strategy.signals import compute_tsmom_signal, compute_signal_strength

lookback = config['strategy']['lookback_months']   # óptimo fijado en el setup
signal_bin  = compute_tsmom_signal(returns, lookback=lookback)
signal_cont = compute_signal_strength(returns, lookback=lookback)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Fracción del tiempo en LONG
pct_long = (signal_bin == 1).mean().sort_values(ascending=False)
axes[0].bar(range(len(pct_long)), pct_long.values, color='steelblue', alpha=0.7, width=1.0)
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1, label='50%')
axes[0].set_title('% del tiempo LONG por ETF')
axes[0].set_xlabel('ETF (ordenado)')
axes[0].legend()

# Señal binaria — SPY
if 'SPY' in signal_bin.columns:
    s = signal_bin['SPY'].dropna()
    axes[1].step(s.index, s.values, where='post', color='darkorange', linewidth=1)
    axes[1].fill_between(s.index, s.values, 0, alpha=0.2, color='darkorange')
    axes[1].set_title('Señal binaria — SPY')
    axes[1].set_ylabel('+1 LONG / -1 SHORT')
    axes[1].set_ylim(-1.5, 1.5)

# Señal continua — SPY
if 'SPY' in signal_cont.columns:
    sc = signal_cont['SPY'].dropna()
    axes[2].plot(sc.index, sc.values, color='navy', linewidth=1)
    axes[2].axhline(0, color='black', linewidth=0.5)
    axes[2].axhline(1, color='green', linestyle=':', linewidth=0.8)
    axes[2].axhline(-1, color='red', linestyle=':', linewidth=0.8)
    axes[2].set_title('Señal continua (Baz) — SPY')
    axes[2].set_ylabel('Z-score clipado [-3, 3]')

plt.tight_layout(); plt.show()
print(f'Promedio % LONG (todas las observaciones): {(signal_bin==1).mean().mean():.1%}')

### ¿Qué es el *net exposure*?

Es uno de los KPIs de arriba del Overview y mide **qué tan direccional está la cartera**: cuánto te quedás
comprado (long) *neto* de lo que estás vendido (short). Acá se calcula por **conteo de posiciones**:

$$\text{net exposure} \;=\; \frac{\#\text{long} \;-\; \#\text{short}}{\#\text{total}}$$

- **+100%** → *todas* las posiciones son long (cartera totalmente comprada).
- **0%** → mitad long, mitad short (neutral al mercado *en conteo*).
- **−100%** → todas short (cartera totalmente vendida).

En el corte de arriba: $123$ long y $10$ short sobre $133$ → $(123-10)/133 = \mathbf{+85\%}$. La cartera está
**fuertemente comprada**, porque casi todo el universo viene con tendencia alcista.

> **No confundir con el *gross exposure*.** Son dos cosas distintas:
> - **Gross** $= |\text{long}| + |\text{short}|$ → cuánto capital total está en juego (el apalancamiento
>   bruto). En el dashboard se normaliza a ~100%.
> - **Net** $= \text{long} - \text{short}$ → cuánto te afecta si *todo* el mercado sube o baja a la vez.
>
> Una cartera puede tener **gross 200% pero net 0%** (100% long + 100% short): mucha plata en juego, pero
> neutral a la dirección del mercado.

**Por qué importa en TSMOM.** El net exposure **no se fija a mano: lo mueve el régimen**. En un mercado
alcista casi todo tiene tendencia positiva → muchos longs → net exposure alto (la estrategia queda
"comprada"). En una crisis se da vuelta → aparecen shorts → el net exposure cae o se vuelve negativo. Por eso
el trend following es *long en los bull markets y short en los bear markets* de forma automática — esa
rotación del net exposure es justo lo que le da su perfil defensivo (la "sonrisa"/convexidad del momentum).

> ⚠️ Ojo con el matiz: este net exposure cuenta **cuántas** posiciones hay de cada lado, no cuánto **riesgo o
> capital** hay de cada lado. Con el vol-scaling (sección B), un short en un activo muy volátil pesa, en
> riesgo, distinto que un long en uno tranquilo. La versión por *conteo* es la que muestra el KPI del Overview
> por ser la más directa de leer.

In [ ]:
# Matriz de señales por clase (igual que el panel 'SEÑAL TSMOM' del Overview)
rows = []
for ac in ASSET_CLASSES:
    tickers = [t for t in last_dir.index if ETF_UNIVERSE.get(t) == ac]
    nl = sum(1 for t in tickers if last_dir.get(t, 0) >= 0)
    rows.append({
        'clase': ac, 'activos': len(tickers),
        'long': nl, 'short': len(tickers) - nl,
        '|fuerza| media': round(last_str[tickers].abs().mean(), 2),
    })
pd.DataFrame(rows).set_index('clase')

### 4.2.bis · `multihorizon`: qué hace y por qué la sacamos del trading

> **Antes que nada — qué dejamos y qué sacamos.** El dashboard mantiene **dos señales operables**:
> `binary` (**Moskowitz 2012**, el signo del momentum) y `strength` (**Baz 2015**, la versión continua
> normalizada por vol). Las dos son competitivas. Lo único que **sacamos** fue `multihorizon`. Es decir,
> seguimos usando aportes de **los dos** papers base: el signo de Moskowitz **y** la señal continua de Baz.

**Qué hace `multihorizon`.** Promedia la señal `strength` calculada en varios lookbacks a la vez, con peso
igual $\tfrac{1}{3}$:

$$s^{\text{multi}}_{i,t} \;=\; \tfrac{1}{3}\Big(s^{\text{str}}_{i,t}(3\text{m}) + s^{\text{str}}_{i,t}(6\text{m}) + s^{\text{str}}_{i,t}(12\text{m})\Big)$$

La idea sale de **Baz et al. (2015)**, que construye su señal CTA como suma ponderada (pesos $\tfrac{1}{3}$)
de la tendencia en **tres escalas de tiempo** distintas. El objetivo es no depender de un único horizonte.

**Por qué la sacamos como opción operable.** La comparación se hace **al óptimo de cada señal** (lookback ×
target vol que maximiza el Sharpe in-sample) — la **misma base que usa el dashboard al abrir** (no a un
12m/10% fijo). Con ese criterio, `multihorizon` **rinde claramente peor que las dos señales que sí dejamos**:

| Señal (pooled, al óptimo) | Sharpe | Params |
|---|---|---|
| `strength` (Baz) — **DEFAULT** | **≈ 0.76** | LB 17m · TV 21% |
| `binary` (Moskowitz) — operable | ≈ 0.69 | LB 11m · TV 39% |
| `multihorizon` (Baz) — **sacada** | ≈ 0.42 | blend 3/6/12 · TV 9% |
| Baz CTA exacto (días, corto) | ≈ 0.06 | TV 23% |
| Baz mecanismo @ 3/6/12m | ≈ 0.24 | TV 23% |

Hay dos razones, y conviene tenerlas claras:

1. **El esquema de Baz está pensado para tendencias rápidas con datos de corto plazo (diarios).** Sus escalas
   son cortas (la más larga, ~96 días ≈ 4.5 meses), apropiadas para un CTA que opera diario. Nosotros
   trabajamos con **rebalanceo mensual** sobre un **universo de ETF**: a esa frecuencia, las escalas cortas
   solo agregan ruido y turnover. La receta exacta de Baz, aun a su mejor target vol, queda en Sharpe **≈0.06**.

2. **Incluso recalibrando el mecanismo a nuestros horizontes (3/6/12 meses), sigue muy por debajo** (≈0.24).
   Mezclar horizontes cortos diluye la señal de **~12-17 meses** —que es el "sweet spot" que documenta
   Moskowitz et al. (2012) y donde caen nuestros óptimos— y sube el turnover sin mejorar el Sharpe.

**Decisión de diseño.** Mantenemos **dos** señales operables —`binary` (Moskowitz) y `strength` (Baz, el
default: mejor Sharpe óptimo y más robusto fuera de muestra)— y **quitamos solo `multihorizon`** del selector
del dashboard. No borramos el código (`compute_multihorizon_signal` sigue en `src/strategy/signals.py`) para
que la prueba sea reproducible y se pueda retomar si cambia el universo o la frecuencia. Esto es justo lo que
pide la consigna: *no replicar el paper, sino adaptarlo con criterio a una estrategia realista*.

In [ ]:
# Comparación al ÓPTIMO de cada señal — la misma base que usa el dashboard (optimize_sharpe al abrir).
from copy import deepcopy
from src.data.loader import load_etf_prices
from src.strategy.signals import compute_crash_filter, compute_multihorizon_signal
from src.strategy.portfolio import apply_position_constraints
from src.backtest.engine import apply_portfolio_vol_scaling
from src.backtest.costs import compute_transaction_costs

# (1) Señales del MOTOR (signal_mode): óptimo conjunto lookback × target_vol, weighting pooled.
def opt_engine(mode):
    c = deepcopy(config); c['strategy']['signal_mode'] = mode; c['strategy']['weighting'] = 'pooled'
    best, _ = optimize_sharpe(returns, c)
    return best

# (2) Señales que NO son signal_mode (multihorizon usa lookbacks fijos; las recetas de Baz, spans
#     fijos): el lookback no aplica, así que optimizamos solo el target_vol sobre la misma grilla.
px = load_etf_prices(config['data']['path'])[list(returns.columns)]

def baz_cta_signal(px, S, L, PW=63, SW=252):
    R = lambda x: x * np.exp(-x**2 / 4) / 0.89               # response function (eq. 32)
    parts = []
    for s, l in zip(S, L):
        x = px.ewm(span=s).mean() - px.ewm(span=l).mean()    # (c) cruce EWMA corto-largo del precio
        y = x.div(px.rolling(PW).std())                      # (d) norm por vol de precio (63d)
        z = y.div(y.rolling(SW).std())                       # (e) norm por std de la señal (252d)
        parts.append(R(z))                                   # (f) response function
    return (sum(parts) / len(parts)).resample('ME').last().shift(1)  # (g) 1/3 + fin de mes, sin lookahead

TVS = [round(0.05 + 0.01 * i, 2) for i in range(36)]         # misma grilla de target_vol que el dashboard
def sharpe_at_tv(signal, tv):
    s = config['strategy']; sig = signal.reindex(returns.index)
    vol = compute_ex_ante_vol(returns, com_months=s['vol_com_months'])
    if s.get('use_crash_filter'):
        sig = sig.mul(compute_crash_filter(returns), axis=0)
    w = apply_position_constraints(compute_vol_scaled_weights(sig, vol, tv), s['max_position_weight'])
    if s.get('use_portfolio_vol_scaling'):
        w = apply_position_constraints(apply_portfolio_vol_scaling(w, returns, s.get('target_portfolio_vol', 0.07), s['vol_com_months']), s['max_position_weight'])
    net = (w * returns).sum(axis=1) - compute_transaction_costs(w, config['transaction_costs']['bid_ask_spread'], config['transaction_costs']['commission_pct'])
    nz = net.ne(0)   # mismo criterio que el motor: arrancar en el primer mes ACTIVO (peso != 0 -> retorno != 0), sin diluir con el warm-up
    return sharpe_ratio(net.loc[nz.idxmax():] if nz.any() else net)
def opt_tv(signal):
    sh = {tv: sharpe_at_tv(signal, tv) for tv in TVS}
    tv = max(sh, key=sh.get); return {'sharpe': sh[tv], 'target_vol': tv}

rows = []
for mode, label in [('binary', 'binary (Moskowitz)  [operable]'),
                    ('strength', 'strength (Baz)  [DEFAULT]')]:
    b = opt_engine(mode)
    rows.append({'señal': label, 'Sharpe (opt)': round(b['sharpe'], 3),
                 'parámetros': f"LB {b['lookback']}m · TV {b['target_vol']*100:.0f}%"})
for label, sig in [('multihorizon 3/6/12m (Baz)  [sacada]', compute_multihorizon_signal(returns, (3, 6, 12))),
                   ('Baz CTA exacto (dias, corto)',         baz_cta_signal(px, (8, 16, 32), (24, 48, 96))),
                   ('Baz mecanismo @ 3/6/12m',              baz_cta_signal(px, (21, 42, 84), (63, 126, 252)))]:
    b = opt_tv(sig)
    tag = 'blend 3/6/12' if 'multihorizon' in label else 'spans fijos'
    rows.append({'señal': label, 'Sharpe (opt)': round(b['sharpe'], 3),
                 'parámetros': f"{tag} · TV {b['target_vol']*100:.0f}%"})
pd.DataFrame(rows).set_index('señal')

### 4.3 `portfolio.py` — De la señal a los pesos (`pooled` vs `class_balanced`)

Tener la señal no basta: hay que decidir **cuánto capital** poner en cada activo. Acá entra el
**vol-scaling** (Moskowitz 2012) y el parámetro **`weighting`** del sidebar.

### Vol-scaling (común a los dos métodos)
Cada activo se dimensiona inverso a su volatilidad, para que **todos aporten riesgo parecido** (no que el
más volátil domine):

$$\text{raw}_{i,t} \;=\; s_{i,t}\cdot\frac{\sigma_{tgt}}{\sigma_{i,t}}$$

El target $\sigma_{tgt}$ (=`target_volatility`, 10%) es el riesgo objetivo *por activo*. Como el riesgo
standalone de la posición es $|w_{i,t}|\cdot\sigma_{i,t}\propto |s_{i,t}|\,\sigma_{tgt}$ (¡no depende de
$\sigma_i$!), **cada activo aporta ~el mismo riesgo**. Falta repartir el presupuesto total — y ahí difieren
los dos métodos:

### `pooled` (default) — 1/N global
Reparte el presupuesto en partes iguales entre **todos** los activos activos:

$$w_{i,t} \;=\; \frac{1}{N_t}\; s_{i,t}\cdot\frac{\sigma_{tgt}}{\sigma_{i,t}}\qquad N_t=\#\text{activos activos}$$

> **Autor.** Moskowitz, Ooi & Pedersen (2012) — es la construcción de cartera del **TSMOM clásico**:
> inverse-vol por activo + promedio $1/N$ sobre todo el universo (en el código, `compute_vol_scaled_weights`).
>
> **Cómo funciona.** Cada activo entra con su peso inverse-vol $s_{i,t}\,\sigma_{tgt}/\sigma_{i,t}$ y todo
> se divide por $N_t$ (la cantidad de activos activos). Es un **$1/N$ global**: el universo entero comparte
> un solo presupuesto, sin distinguir clases.
>
> **Crítica (literatura).** El inverse-vol iguala el riesgo *standalone* de cada activo pero **ignora las
> correlaciones** — no iguala la *contribución* a la varianza. Es justo la crítica de **Maillard, Roncalli &
> Teïletche (2010)** al "naive risk parity" frente al equal-risk-contribution. Atención al matiz: Moskowitz
> et al. **no** sufren esto porque su universo de **futuros ya está balanceado** entre clases (1 instrumento
> por mercado); la concentración del "Problema" de abajo es propia de **nuestro** universo de ETF sesgado a
> equities — es un hallazgo nuestro, no una crítica del paper original.

**Problema:** si cada *nombre* aporta el mismo riesgo y ~67% de los nombres del universo son *equity*
(la mayoría de los ~133 ETF son acciones), entonces **equity se queda con ~70% del presupuesto de riesgo**.
Y como las acciones correlacionan fuerte entre sí (~0.87 intra-equity), su contribución a la *varianza* del
portafolio trepa a ~88%: la diversificación es ilusoria.

> ⚠️ No confundir con el peso en dólares: por el inverse-vol, el *gross* en dólares se concentra en bonos
> (baja vol), pero eso **no** es la apuesta de riesgo. Lo que importa es la contribución de riesgo, que la
> celda de abajo calcula como $|w_i|\cdot\sigma_i$ por clase.

### `class_balanced` — presupuesto de riesgo igual por clase
Da el **mismo presupuesto a cada clase** (equity / bond / commodity / currency) y lo reparte dentro de ella:

$$w_{i,t} \;=\; \frac{1}{N^{\text{clase}}_{i,t}\;\cdot\;K_t}\; s_{i,t}\cdot\frac{\sigma_{tgt}}{\sigma_{i,t}}$$

donde $N^{\text{clase}}_{i,t}$ = activos activos en la clase de $i$, y $K_t$ = nº de clases activas. Esto
replica la diversificación que el TSMOM clásico obtiene de un universo balanceado de **futuros** (Moskowitz
usaba 1 instrumento por mercado, repartido entre equities, bonos, FX y commodities).

> **Autor.** **No** sale de un paper único de TSMOM: es una **decisión de diseño nuestra**, en la línea de
> *risk parity / risk budgeting* (**Maillard, Roncalli & Teïletche 2010**; **Asness, Frazzini & Pedersen
> 2012**, *Leverage Aversion and Risk Parity*). La motivación sí es de Moskowitz et al. (2012): imitar el
> universo **balanceado** de futuros que ellos usaban (en el código, `compute_classbalanced_weights`).
>
> **Cómo funciona.** Arma el mismo peso inverse-vol que `pooled`, pero en vez de dividir por $N_t$ global,
> divide por $N^{\text{clase}}_{i,t}\cdot K_t$: así **cada clase recibe el mismo presupuesto de riesgo** sin
> importar cuántos nombres tenga (equity, con ~89 ETF, recibe lo mismo que currency, con ~7).
>
> **Crítica.** Asume que **todas las clases tienen señal de tendencia igual de explotable**, y acá no es así:
> forzar ~25% del riesgo a FX/commodities (más ruidosas en este universo) le saca presupuesto a la clase con
> mejor señal limpia (equity). Empíricamente **no mejora el Sharpe** acá (ver celda abajo), por eso el default
> quedó en `pooled`. Es una crítica *nuestra*, validada con datos — no del paper.

> En este universo concreto `class_balanced` **no terminó mejorando** el Sharpe (las clases no-equity tienen
> menos señal limpia acá), así que el default quedó en `pooled`. Pero la dejamos como opción y vale para
> *justificar* por qué elegimos `pooled`: lo probamos y comparamos. La celda de abajo muestra el efecto.

In [ ]:
# Comparación pooled vs class_balanced: ¿cómo reparten el RIESGO por clase en el último corte?
# OJO: lo relevante es el RIESGO, no el peso en dólares. Con inverse-vol, el riesgo standalone de
# cada posición es |w_i| * sigma_i. (El peso en dólares se concentra en bonos de baja vol, pero eso
# no es la 'apuesta de riesgo': cada nombre aporta ~ el mismo riesgo, así que la clase con más
# nombres domina.)
vol = compute_ex_ante_vol(returns, com_months=config['strategy']['vol_com_months'])
tv  = config['strategy']['target_volatility']

w_pooled = compute_vol_scaled_weights(strength,   vol, target_vol=tv)
w_cb     = compute_classbalanced_weights(strength, vol, target_vol=tv)

def risk_by_class(w):
    rc = (w.abs() * vol).iloc[-1]   # riesgo standalone por activo = |w_i| * sigma_i
    s = pd.Series({ac: rc[[t for t in rc.index if ETF_UNIVERSE.get(t) == ac]].sum()
                   for ac in ASSET_CLASSES})
    return (s / s.sum() * 100).round(1)

print("Share de NOMBRES por clase (%):  equity domina el universo")
names = pd.Series({ac: sum(1 for t in returns.columns if ETF_UNIVERSE.get(t) == ac)
                   for ac in ASSET_CLASSES})
print((names / names.sum() * 100).round(1).to_string(), "\n")

pd.DataFrame({
    'pooled  (% riesgo)':         risk_by_class(w_pooled),
    'class_balanced  (% riesgo)': risk_by_class(w_cb),
})

In [ ]:
from src.strategy.portfolio import compute_vol_scaled_weights, apply_position_constraints

weights = apply_position_constraints(
    compute_vol_scaled_weights(signal_bin, vol, target_vol=config['strategy']['target_volatility']),
    max_weight=0.15
)

last_w = weights.dropna(how='all').iloc[-1].sort_values()
last_date = weights.index[-1].strftime('%Y-%m')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['salmon' if v < 0 else 'steelblue' for v in last_w.values]
axes[0].barh(range(len(last_w)), last_w.values * 100, color=colors, alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_yticks(range(0, len(last_w), 5))
axes[0].set_yticklabels(last_w.index[::5])
axes[0].set_title(f'Pesos — {last_date}')
axes[0].set_xlabel('Peso (%)')

axes[1].plot(weights.sum(axis=1).index, weights.sum(axis=1).values * 100,
             label='Exposición neta', color='navy')
axes[1].plot(weights.abs().sum(axis=1).index, weights.abs().sum(axis=1).values * 100,
             label='Exposición bruta', color='darkorange', linestyle='--')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Exposición del portafolio (%)')
axes[1].legend()

plt.tight_layout(); plt.show()

# Verificar contribución de riesgo
risk_contrib = (weights.abs() * vol).dropna(how='all').iloc[-1].sort_values(ascending=False)
print(f'Top 5 contribuciones al riesgo (mes {last_date}):')
print((risk_contrib.head(5) * 100).round(3).to_string())

### 4.4 El NAV y los KPIs del Overview (pipeline completo)

**Ojo con una distinción importante:** la *matriz de señales* de arriba muestra la señal "cruda" (signo e
intensidad por activo). Pero el **NAV** y los **KPIs** salen del **backtest completo**, que además de la señal
y el vol-scaling aplica dos capas más (las vemos en detalle en *Backtest* y *Robustness*):

- **Filtro de crash** (Daniel & Moskowitz 2016): si el mercado está en *bear* (retorno 12m del SPY < 0) **y**
  con vol disparada (vol corta > 1.2× vol larga), recorta la señal a la mitad (×0.5). Evita los
  "momentum crashes" (los rebotes violentos tras un derrumbe, que castigan al momentum).
- **Vol-scaling del portafolio** (Moreira & Muir 2017): escala la cartera *entera* para apuntar a un riesgo
  objetivo `target_portfolio_vol` (7%), con tope de apalancamiento 2×. Sube exposición cuando hay calma y la
  baja cuando la vol realizada sube.

El motor (`BacktestEngine`) y la ejecución en vivo en IBKR usan **exactamente los mismos pesos**
(`build_scaled_weights`), así que lo que se backtestea es lo que se opera.

In [ ]:
# KPIs del Overview, con el pipeline COMPLETO (señal + crash filter + vol-scaling de portafolio)
res = BacktestEngine(config).run(returns, use_signal_strength=(config['strategy']['signal_mode'] == 'strength'))
pr  = res.portfolio_returns        # retornos netos de costos
eq  = (1 + pr).cumprod()           # NAV (base 1.00x)

print(f"Sharpe (IS)   : {sharpe_ratio(pr):.2f}        (excess T-bill 1m)")
print(f"Ann. Return   : {annualized_return(pr):+.2%}      (neto de costos)")
print(f"Ann. Vol      : {annualized_volatility(pr):.2%}       (target portfolio {config['strategy']['target_portfolio_vol']:.0%})")
print(f"Max Drawdown  : {max_drawdown(pr):.2%}")
print(f"Calmar        : {calmar_ratio(pr):.2f}")
print(f"NAV final     : {eq.iloc[-1]:.3f}x   ({eq.iloc[-1]-1:+.1%} desde inicio)")

In [ ]:
# Vista rápida del NAV (el 'sparkline' del panel NAV es esta misma curva)
eq.plot(figsize=(9, 3), title=f"NAV {returns.index[0].date()} -> {returns.index[-1].date()}  (base 1.00x)", grid=True);

### Checklist de la sección Overview
- [ ] Entiendo la señal: `binary` (signo) vs `strength` (continua normalizada), y el rol del **lookback** (12m).
- [ ] Entiendo la **vol ex-ante** (EWMA, com=3) y por qué normalizamos por riesgo (que todos aporten igual).
- [ ] Entiendo **`pooled`** (1/N global) vs **`class_balanced`** (presupuesto igual por clase) y por qué el default es `pooled`.
- [ ] Entiendo que el **NAV/KPIs** suman crash filter + vol-scaling de portafolio (no solo la señal cruda).
- [ ] Net exposure, # long/short y NAV del notebook coinciden con la terminal.


---
<a id='s5'></a>
## 5. Backtesting framework — `src/backtest/`

La página **Backtest** responde la pregunta central del TP: **¿la estrategia hubiera funcionado?** Toma los
pesos del motor (`BacktestEngine` — los mismos que se operan en vivo en IBKR), los aplica mes a mes sobre los
retornos reales del universo, **descuenta costos** y resume el resultado. Tiene cinco bloques:

1. **KPIs** (tira de arriba) — Sharpe, retorno anualizado, volatilidad, max drawdown y Calmar.
2. **Curva de capital** (escala log) — el NAV **neto** vs el **bruto** (sin costos).
3. **Drawdown** — cuánto está la cartera por debajo de su máximo histórico (*underwater*).
4. **Tabla de métricas** — las mismas medidas + Sortino, hit rate, turnover y nº de meses.
5. **Heatmap de retornos mensuales** — cada mes coloreado (verde = ganancia, rojo = pérdida).

Lo recorremos en tres pasos: primero **de dónde sale cada retorno** (A), después **cómo se calcula cada
métrica** (B), y por último **qué dice cada gráfico** (C). Todos los números salen del mismo backend que el
dashboard.

### 5.1 `costs.py` — De dónde sale cada retorno: bruto, costos y neto

El motor (`src/backtest/engine.py`) arma la serie de retornos del portafolio en tres pasos:

**1) Retorno bruto** — la posición de cada activo por su retorno del mes, sumado sobre todo el universo:

$$r^{\text{bruto}}_t \;=\; \sum_i w_{i,t}\, r_{i,t}$$

Clave de *timing*: $w_{i,t}$ se decide con información **hasta $t-1$** (la señal lleva `shift(1)`) y recién
después gana el retorno del mes $t$. Así **no hay look-ahead bias**: nunca usás el dato del mes para decidir
la posición de ese mismo mes.

**2) Costos de transacción** — cada vez que cambiás los pesos pagás *spread* + comisión sobre lo que moviste
(`src/backtest/costs.py`):

$$\text{costo}_t \;=\; \underbrace{\sum_i \lvert\Delta w_{i,t}\rvert}_{\text{turnover}_t}\;\cdot\;\Big(\tfrac{\text{spread}}{2} + \text{comisión}\Big)$$

Vale la pena abrir **cada valor** de esa fórmula, porque cada uno modela un costo real distinto:

- **$\text{turnover}_t = \sum_i \lvert\Delta w_{i,t}\rvert$** — *cuánto rotó la cartera ese mes*: la suma de
  los cambios **absolutos** de peso, activo por activo. Si un ETF pasa de $0\% \to 5\%$ (comprás 5) y otro de
  $3\% \to 0\%$ (vendés 3), el turnover es $5\% + 3\% = 8\%$. Es la **base imponible** del costo: solo pagás
  por lo que efectivamente comprás o vendés, no por lo que dejás quieto. El valor absoluto hace que comprar y
  vender cuesten igual (los dos mueven mercado).

- **$\dfrac{\text{spread}}{2}$** — acá está el *"¿por qué dividido 2?"*. El **bid-ask spread** es la brecha
  entre el precio al que el mercado te **vende** (ask, más caro) y al que te **compra** (bid, más barato). El
  "precio justo" es el punto medio (*mid*). Cuando hacés **una** operación, no cruzás el spread entero: al
  comprar pagás medio spread por encima del mid, y al vender cobrás medio spread por debajo. Es decir, **una
  transacción cuesta la mitad del spread cotizado** → $\text{spread}/2$ (el costo *one-way*; el spread entero
  recién lo pagás si comprás *y* después vendés, *round-trip*). Con `bid_ask_spread = 10 bps` del `config`,
  eso es **5 bps** por operación.

- **$\text{comisión}$** — el fee del broker, como % del monto operado. A diferencia del spread, esto **no** se
  divide: lo cobran completo en cada trade. Con `commission_pct = 5 bps`, son **5 bps**.

- **Suma** → cada unidad de turnover cuesta $\tfrac{\text{spread}}{2} + \text{comisión} = 5 + 5 = \textbf{10
  bps}$ *one-way*. Más rebalanceo ⇒ más turnover ⇒ más costo: por eso vigilamos el turnover (y por eso una
  señal que *flipea* mucho, como `binary` en activos planos, es cara).

> **Supuestos del modelo (para ser honestos).** Es un modelo de costos **proporcional y lineal**: asume
> spread y comisión fijos para todos los ETF y todos los tamaños de orden. En la realidad el spread varía por
> liquidez y se *ensancha* en órdenes grandes (*market impact*), pero como operamos ETF líquidos con
> rebalanceo mensual, un costo plano de 10 bps one-way es una aproximación conservadora y estándar.

**3) Retorno neto** — lo que de verdad te queda:

$$r^{\text{neto}}_t \;=\; r^{\text{bruto}}_t - \text{costo}_t$$

> **Todas las métricas de performance se calculan sobre el retorno NETO.** La diferencia bruto − neto es el
> "peaje" de operar; la celda de abajo lo cuantifica.

In [ ]:
# Backtest COMPLETO (mismo motor que el dashboard y que la ejecución en vivo en IBKR)
res   = BacktestEngine(config).run(returns, use_signal_strength=(config['strategy']['signal_mode'] == 'strength'))
gross = res.gross_returns       # retorno mensual ANTES de costos
net   = res.portfolio_returns   # retorno mensual NETO (gross - costos)

tc = config['transaction_costs']
per_unit = tc['bid_ask_spread'] / 2 + tc['commission_pct']
print(f"Costo por unidad de turnover : spread/2 + comisión = "
      f"{tc['bid_ask_spread']/2*1e4:.0f} + {tc['commission_pct']*1e4:.0f} = {per_unit*1e4:.0f} bps (one-way)")
print(f"Turnover mensual promedio    : {res.weights.diff().abs().sum(axis=1).mean():.1%}")
print(f"Costo anual promedio (drag)  : {res.costs.mean()*12*100:.2f}%\n")
print(f"Ann. Return BRUTO  : {annualized_return(gross):+.2%}")
print(f"Ann. Return NETO   : {annualized_return(net):+.2%}")
print(f"Peaje de operar    : {(annualized_return(gross)-annualized_return(net))*100:.2f} pp por año")

### 5.2 `metrics.py` — Las métricas de performance

Todas se calculan sobre el retorno **mensual neto** y se anualizan con factor $12$ (o $\sqrt{12}$ para los
desvíos). Convención del proyecto: el Sharpe y el Sortino se miden sobre el **retorno en exceso de la T-bill a 1 mes** ($rf$ variable en el tiempo; ver §D). El retorno, la volatilidad y el drawdown se reportan sobre el retorno total. (Sharpe (1994):
el ratio debe medirse sobre un retorno diferencial / en exceso, no sobre el retorno total).

| Métrica | Fórmula | Qué dice |
|---|---|---|
| **Annualized Return** | $\big(\prod_t (1+r_t)\big)^{12/n} - 1$ | retorno compuesto por año |
| **Annualized Volatility** | $\operatorname{std}(r_t)\cdot\sqrt{12}$ | cuánto oscila por año (riesgo total) |
| **Sharpe Ratio** | $\dfrac{\overline{r}}{\operatorname{std}(r)}\cdot\sqrt{12}$ | retorno por unidad de **riesgo total** |
| **Sortino Ratio** | $\dfrac{\overline{r}}{\operatorname{std}(r_t\,<\,0)}\cdot\sqrt{12}$ | como Sharpe, pero castiga **solo las caídas** |
| **Max Drawdown** | $\min_t \dfrac{\text{NAV}_t - \max_{s\le t}\text{NAV}_s}{\max_{s\le t}\text{NAV}_s}$ | la peor caída pico-a-valle |
| **Calmar Ratio** | $\dfrac{\text{Ann. Return}}{\lvert\text{Max Drawdown}\rvert}$ | retorno por unidad de **peor caída** |
| **Hit Rate** | $\operatorname{mean}(r_t > 0)$ | % de meses ganadores |
| **Avg Monthly Turnover** | $\operatorname{mean}_t \sum_i \lvert\Delta w_{i,t}\rvert$ | cuánto rota la cartera por mes |

> **Sharpe vs Sortino vs Calmar** miran el riesgo de forma distinta: Sharpe usa *toda* la volatilidad (premia
> y castiga los movimientos hacia arriba y hacia abajo por igual); Sortino solo la volatilidad *a la baja* (lo
> que de verdad duele); y Calmar lo reduce al peor momento histórico (el max drawdown). Que **Sortino > Sharpe**
> indica que la cartera tiene más volatilidad "buena" (al alza) que "mala" (a la baja).

In [ ]:
# Tabla de MÉTRICAS DE PERFORMANCE — exactamente las que muestra el panel del dashboard
from src.backtest.metrics import sortino_ratio, hit_rate, turnover

r = res.portfolio_returns   # retorno mensual NETO
tabla = pd.DataFrame(
    {'valor': [
        f"{annualized_return(r):+.2%}",
        f"{annualized_volatility(r):.2%}",
        f"{sharpe_ratio(r):.2f}",
        f"{sortino_ratio(r):.2f}",
        f"{max_drawdown(r):.2%}",
        f"{calmar_ratio(r):.2f}",
        f"{hit_rate(r):.1%}",
        f"{turnover(res.weights):.1%}",
        f"{len(r)}",
    ]},
    index=['Annualized Return', 'Annualized Volatility', 'Sharpe Ratio', 'Sortino Ratio',
           'Max Drawdown', 'Calmar Ratio', 'Hit Rate', 'Avg Monthly Turnover', 'Num Months'],
)
tabla.index.name = 'métrica'
tabla

### 5.3 `engine.py` — Qué dice cada gráfico

**Curva de capital (escala log).** El NAV (1.00× al inicio) compuesto en el tiempo, en **neto** (verde) vs
**bruto** (gris punteado). Va en **escala logarítmica** a propósito: en log, una pendiente constante =
**tasa de crecimiento constante**, y un +10% se ve igual de "alto" arrastrando \$1 que arrastrando \$2M. Sin
log, los últimos años aplastarían visualmente a los primeros. La brecha que se abre entre neto y bruto es el
**costo acumulado** de operar.

**Drawdown (underwater).** En cada mes, cuánto está el NAV **por debajo de su máximo histórico**:
$\text{dd}_t = (\text{NAV}_t - \max_{s\le t}\text{NAV}_s)\,/\,\max_{s\le t}\text{NAV}_s$. Siempre $\le 0$;
vuelve a $0$ cuando la cartera hace un nuevo máximo. El punto más profundo es el **Max Drawdown**. Sirve para
ver no solo *cuánto* se perdió, sino *cuánto tardó en recuperarse* (el ancho del valle).

**Heatmap de retornos mensuales.** Cada celda es el retorno de un mes (filas = años, columnas = meses),
coloreada verde (ganancia) / rojo (pérdida). De un vistazo se ven la consistencia, los meses malos
concentrados (las crisis aparecen como franjas rojas) y la estacionalidad.

In [ ]:
# Curva de capital en escala LOG: NAV neto (con costos) vs bruto (sin costos)
import matplotlib.pyplot as plt
cum_net   = (1 + res.portfolio_returns).cumprod()   # NAV neto (base 1.00x)
cum_gross = (1 + res.gross_returns).cumprod()        # NAV bruto (sin costos)

ax = cum_gross.plot(figsize=(9, 3.2), logy=True, color='gray', ls='--', lw=1.2,
                    label='Bruto (sin costos)')
cum_net.plot(ax=ax, logy=True, color='green', lw=2, label='Neto (con costos)')
ax.set_title('Curva de capital — escala log · neto vs bruto')
ax.set_ylabel('NAV (log, base 1.00x)'); ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()

In [ ]:
# Drawdown (underwater %): cuánto está el NAV por debajo de su máximo histórico
dd = (cum_net - cum_net.cummax()) / cum_net.cummax()
ax = (dd * 100).plot(figsize=(9, 2.6), color='crimson', lw=1.2)
ax.fill_between(dd.index, dd.values * 100, 0, color='crimson', alpha=0.15)
ax.set_title(f'Drawdown (underwater %) · peor caída = {dd.min()*100:.1f}%')
ax.set_ylabel('%'); ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
# Heatmap de retornos mensuales (mismo pivot que el panel 'RETORNOS MENSUALES')
r = res.portfolio_returns
piv = (pd.DataFrame({'year': r.index.year, 'month': r.index.month, 'ret': r.values})
         .pivot(index='year', columns='month', values='ret')
         .reindex(columns=range(1, 13)))
piv.columns = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
(piv * 100).style \
    .background_gradient(cmap='RdYlGn', axis=None, vmin=-8, vmax=8) \
    .format('{:.1f}', na_rep='·') \
    .set_caption('Retornos mensuales (%) · verde = ganancia, rojo = pérdida')

### 5.4 `riskfree.py` — La tasa libre de riesgo (exceso sobre la T-bill a 1 mes)

**Decisión del proyecto:** el Sharpe y el Sortino se miden sobre el **retorno en EXCESO de la T-bill a 1 mes**
(tasa real, variable mes a mes). El retorno, la volatilidad y el drawdown se siguen reportando sobre el
retorno total. Esta sección justifica por qué — y por qué *no* dejamos $rf = 0$.

**Por qué en exceso (y no $rf=0$).** Sharpe (1994) es taxativo: el ratio debe computarse sobre un retorno
*diferencial / en exceso*, "o pierde su razón de ser". Los papers de TSMOM (Moskowitz-Ooi-Pedersen 2012;
Baz 2015; AQR) usan $rf=0$ **efectivo** solo porque operan **futuros colateralizados**, cuyo retorno *ya* es
en exceso de financiamiento. Pero **nosotros operamos ETF cash**: la cartera es *fully-funded* y **apalancada**
(gross promedio ~145%), así que corresponde **restar la tasa de financiamiento**. Como los retornos son
*totales* (precios ajustados), el sleeve cash-like (SGOV/SHY/BIL, ~27% del libro) que ya rinde ≈$rf$ **no
genera doble-conteo**: restar $rf$ simplemente le deja exceso ≈0, que es lo contablemente correcto.

**Por qué 1 mes.** El $rf$ del Sharpe matchea el **período de tenencia/medición**, no el lookback de la señal.
Rebalanceamos y medimos **mensual** → la T-bill a 1 mes es el "one-period riskless asset" de Sharpe (1994). Un
plazo más largo (2y, 5y) **no** sería libre de riesgo sobre un mes (tiene riesgo de duración) → sería un error.

**De dónde sale.** Serie oficial del **Tesoro de EE.UU.** (T-bill 4 semanas, coupon-equivalent = `DGS1MO`),
2012-2026, cacheada en `Data/riskfree_1m.parquet` (no depende de la red en cada corrida). Promedió ~2.0% en el
período activo: ~0% en 2014-2021 (ZIRP), pico ~5.4% en 2023, y **3.69%** al cierre (may-2026, la tasa actual).

> **$rf=0$ queda solo como comparación con la literatura.** No es nuestra métrica; lo mostramos al lado para
> comparar con los papers de managed futures (cuyos futuros colateralizados ya dan retornos en exceso). En
> nuestro caso: **Sharpe excess ≈ 0.76** (la métrica del proyecto) vs **rf=0 ≈ 1.00** (base comparable).

Fuentes: Sharpe (1994, Stanford); Goetzmann-Ingersoll-Spiegel-Welch (NBER w9116); Moskowitz-Ooi-Pedersen
(2012, JFE); Hurst-Ooi-Pedersen (AQR/JPM 2017); U.S. Treasury / Fed H.15 (DGS1MO).

In [ ]:
# La rf del proyecto: T-bill a 1 mes real (Tesoro EE.UU.), variable en el tiempo
from src.data.riskfree import load_tbill_1m_monthly
from src.backtest.metrics import sharpe_ratio, sortino_ratio

rf = load_tbill_1m_monthly(r.index)          # rf mensual real, alineada (cacheada en Data/)
print("Fuente rf:", rf.attrs.get("source"))
print(f"T-bill 1m anual: prom {rf.mean()*12*100:.2f}%  |  actual ({r.index[-1].date()}) {rf.iloc[-1]*12*100:.2f}%\n")

tabla_rf = pd.DataFrame({
    "Sharpe":  [sharpe_ratio(r),  sharpe_ratio(r, rf=0.0)],
    "Sortino": [sortino_ratio(r), sortino_ratio(r, rf=0.0)],
}, index=["EXCESS T-bill 1m (métrica del proyecto)", "rf = 0 (solo comparación c/ literatura)"]).round(3)
display(tabla_rf)

### Checklist de la sección Backtest
- [ ] Entiendo **bruto vs neto**: $r^{\text{bruto}}=\sum_i w_{i,t}\,r_{i,t}$, costo $=$ turnover $\times$ (spread/2 + comisión), neto $=$ bruto $-$ costo.
- [ ] Entiendo el *timing* `shift(1)` → **sin look-ahead** (la posición de $t$ se decide con datos hasta $t-1$).
- [ ] Sé qué mide cada métrica y por qué **Sharpe / Sortino / Calmar** ven el riesgo distinto.
- [ ] Entiendo por qué la curva va en **escala log** y qué es el **drawdown** (underwater).
- [ ] Los KPIs y la tabla coinciden con la terminal (mismo motor, retorno neto, **exceso sobre T-bill 1m**).


---
<a id='s6'></a>
## 6. Tests de robustez — `src/robustness/`

Un buen backtest no alcanza: hay que mostrar que el resultado **no es producto de la suerte** — ni de haber
acertado un parámetro, ni de un período particular, ni de mirar los mismos datos con que se calibró. La página
**Tests de Robustez** ataca esas tres dudas, una por pestaña:

1. **Sensibilidad paramétrica** (A) — ¿el Sharpe se cae si muevo el lookback, los costos o el target vol? Una
   estrategia robusta degrada *suave*; no tiene un óptimo "filo de cuchillo".
2. **Stress test** (B) — ¿cómo se comportó en las crisis (COVID 2020, shock de tasas 2022), y contra buy & hold?
3. **Out-of-sample** (C) — ¿sobrevive fuera de la muestra? Walk-forward **re-optimizando los parámetros en cada
   ventana** y recorriendo **toda** la línea de tiempo.

Todo sale del mismo backend (`src/robustness/*`), que **re-corre el motor real** en cada variante. Es la
sección que convierte "me dio buen Sharpe" en "el resultado es creíble". Cada celda de código imprime, además
de las tablas, una **"lectura" automática** que se recalcula con los datos — así el análisis nunca queda
desincronizado de los números.

### 6.1 `sensitivity.py` — Sensibilidad paramétrica

**Cómo se calcula.** Las tres funciones de `src/robustness/sensitivity.py` hacen lo mismo: para cada valor de
una grilla, `deepcopy(config)` -> cambian **un solo** parámetro -> re-corren el **backtest completo**
(`BacktestEngine`, idéntico al del dashboard) -> guardan `sharpe`, `ann_ret`, `max_dd` del retorno neto. El
resto del config queda fijo en el óptimo (**17m / 21%**). Es, literalmente, una *rebanada 1-D* del espacio de
parámetros alrededor del punto de operación: muevo un eje y miro si el Sharpe aguanta.

- **`lookback_sensitivity`** — barre **todos** los meses de tendencia de 3 a 24 (una barra por mes, sin huecos).
- **`cost_sensitivity`** — sube el spread bid-ask `[0, 10, 50, 100, 200] bps` (real = 10 bps).
- **`target_vol_sensitivity`** — el riesgo objetivo *por activo* `[5%…20%]`.

**Qué buscamos:** que el Sharpe degrade *suave* (no un óptimo "filo de cuchillo") y se mantenga **positivo en
todo el rango**. Que dependa de clavar un valor exacto sería señal de overfitting.

**Lectura de los resultados (corrida actual):**

- **Lookback** — el Sharpe es **positivo en todo el rango** (~0.27 en 3m hasta ~0.66 en la zona 9-18m). No
  hay colapso: incluso el peor horizonte (3m, puro ruido) da ~0.47. El pico cae en la zona **~9–18m** —el
  *sweet spot* de Moskowitz et al. (2012)— aunque no es una meseta perfecta (hay un bache en 12–15m ~0.55–0.67
  y repuntes en 9m y 18m). Conclusión: no vive de un lookback puntual, pero tampoco es plana; conviene declararlo.
- **Costos** — a los **10 bps reales** el Sharpe es ~1.00; aguanta con Sharpe>=0.5 hasta **~100 bps (10x lo
  real)** y recién se desarma cerca de 200 bps (~0.27). El resultado **no depende de fricciones irrealmente bajas**.
- **Target vol** — casi **plano** (~0.89 -> 1.00 entre 5% y 20%). Tiene sentido: arriba actúa el **vol-scaling
  del portafolio** (Moreira-Muir, objetivo 7%), que reescala la cartera entera; mover el target *por activo*
  cambia el apalancamiento intermedio pero no la mezcla de señal. Esa **insensibilidad es, en sí, robustez**.

In [ ]:
# Sensibilidad: re-corre el backtest cambiando UN parámetro por vez (resto en el óptimo)
from IPython.display import display
from src.robustness.sensitivity import lookback_sensitivity, cost_sensitivity, target_vol_sensitivity
import matplotlib.pyplot as plt

lb_grid = list(range(3, 25))                                    # TODOS los meses (sin huecos)
lb = lookback_sensitivity(returns, config, lookbacks=lb_grid)   # incluye el lookback operado
cs = cost_sensitivity(returns, config)                          # spreads [0,10,50,100,200] bps
tv = target_vol_sensitivity(returns, config)                    # target vol [5%..20%]

print("Sharpe por LOOKBACK (meses) — resto del config en el óptimo:"); display(lb.round(3))
print("Sharpe vs COSTOS (spread bid-ask; real = 0.001 = 10 bps):"); display(cs.round(3))
print("Sensibilidad al TARGET VOL (riesgo objetivo por activo):"); display(tv.round(3))

ax = lb['sharpe'].plot(kind='bar', figsize=(8, 2.6),
        color=['green' if v >= 0 else 'crimson' for v in lb['sharpe']])
ax.set_title('Sharpe por lookback (resto en el óptimo)')
ax.set_xlabel('lookback (meses)'); ax.set_ylabel('Sharpe'); ax.axhline(0, color='gray', lw=0.8)
plt.tight_layout(); plt.show()

# --- Lectura automática (se recalcula con los datos) ---
costo_ok = cs[cs['sharpe'] >= 0.5].index.max()        # mayor spread que todavía da Sharpe >= 0.5
print("LECTURA — Sensibilidad")
print(f"  Lookback : Sharpe {lb['sharpe'].min():.2f} ({lb['sharpe'].idxmin()}m) a {lb['sharpe'].max():.2f} "
      f"({lb['sharpe'].idxmax()}m) - {'TODOS > 0 -> sin filo de cuchillo' if (lb['sharpe']>0).all() else 'alguno < 0 -> cuidado'}")
print(f"  Costos   : a 10bps Sharpe {cs.loc[0.001,'sharpe']:.2f}; aguanta Sharpe>=0.5 hasta {costo_ok*1e4:.0f}bps "
      f"({costo_ok/0.001:.0f}x lo real)")
print(f"  TargetVol: rango {tv['sharpe'].min():.2f}-{tv['sharpe'].max():.2f} en 5%-20% -> casi plano "
      f"(lo fija el vol-scaling de portafolio al 7%)")

### 6.2 `stress_test.py` — Stress test (performance en crisis)

**Cómo se calcula.** `crises_in_window(r.index)` filtra `CRISIS_PERIODS` (GFC 2008, Taper 2013, COVID 2020,
shock de tasas 2022) a las que **solapan con la ventana activa** del backtest; `analyze_crisis_performance`
corta el retorno neto en cada ventana (`portfolio_returns.loc[inicio:fin]`) y anualiza retorno, vol y Sharpe,
y calcula el max drawdown de ese tramo. El equity sombrea **exactamente** esas ventanas (las mismas que la
tabla -> no se pinta sobre vacío).

La pregunta del trend following: **¿defiende en los derrumbes?** La teoría dice que sí — en un bear market la
señal se da vuelta a short y la cartera puede ganar mientras el mercado cae (la "convexidad" del momentum).

> ⚠️ **Cobertura honesta.** La estrategia recién opera desde **2014-11** (warm-up de la señal, §02), así que
> **GFC 2008 y Taper 2013 quedan fuera** y no se reportan. Quedan **COVID 2020** y **shock de tasas 2022**.

**Lectura (corrida actual):**

- **COVID 2020** — Sharpe ~**+0.45**, retorno anualizado ~**+3.3%**, maxDD acotado (~−5.6%). **Defendió**: la
  rotación a defensivos/short amortiguó el desplome de marzo. Es el comportamiento que justifica tener trend
  following en la cartera.
- **Shock de tasas 2022** — Sharpe ~**−1.33**, retorno ~**−5.6%**. **El peor año.** Bonos y acciones cayeron
  juntos y los cambios bruscos de régimen (rebotes violentos) castigan al momentum (*whipsaws*). Está bien
  mostrarlo: no todo da verde, y es justo el escenario donde el trend following sufre.

In [ ]:
# Stress test: equity con crisis sombreadas (solo las que solapan la ventana) + tabla por crisis
from src.robustness.stress_test import analyze_crisis_performance, crises_in_window, CRISIS_PERIODS
from src.backtest.engine import BacktestEngine

res = BacktestEngine(config).run(returns)
eq = (1 + res.portfolio_returns).cumprod()
activas = crises_in_window(res.portfolio_returns.index)   # mismas crisis que muestra el dashboard

ax = eq.plot(figsize=(9, 3.2), logy=True, color='green', lw=1.8)
ax.set_xlim(eq.index[0], eq.index[-1])
for name, (s, e) in activas.items():
    s, e = pd.Timestamp(s), pd.Timestamp(e)
    ax.axvspan(max(s, eq.index[0]), min(e, eq.index[-1]), color='red', alpha=0.12)
    ax.text(max(s, eq.index[0]), eq.max(), f" {name}", fontsize=7, color='gray', va='top')
ax.set_title('Equity curve (log) — crisis sombreadas'); ax.set_ylabel('NAV (log)')
ax.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

fuera = [c for c in CRISIS_PERIODS if c not in activas]
print(f"Crisis dentro de la ventana activa: {list(activas)}  |  fuera (no se reportan): {fuera}")
cp = analyze_crisis_performance(res.portfolio_returns)
display(cp)

print("LECTURA — Crisis")
for period, row in cp.iterrows():
    sh = float(row['Sharpe'])
    print(f"  {period:16s}: Sharpe {row['Sharpe']:>6} - ret {row['Ann Return']:>7} - maxDD {row['Max DD']:>7}"
          f"  -> {'DEFENDIO' if sh > 0 else 'sufrio'}")

### ¿El −5.6% de 2022 es "bueno"? Comparación contra buy & hold

Un retorno negativo en una crisis no dice nada por sí solo: hay que leerlo **contra lo que habrías tenido sin
la estrategia**. La tabla compara, en cada crisis activa, el **retorno total** y el **peor drawdown** de la
estrategia contra tres alternativas pasivas: comprar y mantener el S&P 500 (`SPY`), un **60/40** clásico (60%
acciones / 40% bonos `AGG`) y el **universo entero equal-weight**.

**Cómo se calcula:** para cada ventana de crisis, retorno compuesto $\prod_t (1+r_t)-1$ y el peor drawdown del
tramo, tanto para la estrategia (`res.portfolio_returns`) como para cada benchmark armado con los mismos
retornos mensuales del universo.

**Lectura (corrida actual):**

- **Shock de tasas 2022** — la estrategia perdió **−5.6%** mientras el S&P cayó **≈−18%** y un 60/40 **≈−16%**
  (fue uno de los peores años del 60/40 en un siglo: acciones *y* bonos cayeron juntas, así que la
  diversificación clásica no protegió). El trend following dio vuelta las señales a **short** y amortiguó
  ~12 pp del golpe, con un drawdown de solo −5.9% vs −16/−20% del buy & hold. **No ganó plata, pero convirtió
  un −18% en un −5.6%** — esa es su función defensiva (convexidad).
- **COVID 2020** — caso más matizado: la ventana (ene–sep) incluye la recuperación en V, así que el S&P
  terminó *arriba* (≈+5.5%) y la estrategia un poco menos (≈+2.5%). Pero el **drawdown** cuenta la otra mitad:
  −5.6% de la estrategia vs **≈−19%** del S&P. Protegió fuerte en el desplome de marzo, aunque —como buen
  trend follower— tardó en re-engancharse al rebote (cede algo de la suba rápida): el precio de la suavidad.

> Por eso el Sharpe negativo de 2022 (−1.33) **no** hay que leerlo como catástrofe: la caída real fue chica y
> mucho menor que la del mercado. El stress test no mide 'ganar en la crisis', sino **perder mucho menos** que
> estar pasivo.

In [ ]:
# ¿El retorno en crisis es "bueno"? -> leerlo contra buy & hold (SPY, 60/40, universo eq-weight)
bench = {
    'TSMOM (estrategia)':       res.portfolio_returns,
    'B&H S&P 500 (SPY)':        returns['SPY'],
    'B&H 60/40 (SPY/AGG)':      0.6 * returns['SPY'] + 0.4 * returns['AGG'],
    'B&H universo (eq-weight)': returns.mean(axis=1),
}
def _crisis_stats(s, a, b):
    w = s.loc[a:b]
    cum = (1 + w).prod() - 1                                            # retorno total del tramo
    dd  = ((1 + w).cumprod() / (1 + w).cumprod().cummax() - 1).min()    # peor drawdown del tramo
    return cum, dd

filas = []
for cn, (a, b) in activas.items():                                     # mismas crisis que el panel de arriba
    for lab, s in bench.items():
        cum, dd = _crisis_stats(s, a, b)
        filas.append({'Crisis': cn, 'Cartera': lab,
                      'Retorno': f"{cum*100:+.1f}%", 'Max DD': f"{dd*100:.1f}%"})
display(pd.DataFrame(filas).set_index(['Crisis', 'Cartera']))

print("LECTURA — TSMOM vs buy & hold (retorno / peor drawdown del tramo)")
for cn, (a, b) in activas.items():
    sr, sd = _crisis_stats(bench['TSMOM (estrategia)'], a, b)
    pr, pq = _crisis_stats(bench['B&H S&P 500 (SPY)'], a, b)
    print(f"  {cn:16s}: estrategia {sr*100:+5.1f}% / DD {sd*100:5.1f}%   vs   "
          f"S&P {pr*100:+5.1f}% / DD {pq*100:5.1f}%   -> menor caida: estrategia")

### 6.3 `out_of_sample.py` — Validación out-of-sample

El test más exigente: ¿la estrategia funciona en datos que **no** se usaron para elegir los parámetros? Lo
medimos con un **walk-forward anclado y re-optimizado**, que recorre **toda** la línea de tiempo.

**Cómo se calcula** (`walk_forward`) — en cada ventana:
1. con **todos los datos hasta `train_end`** se re-corre `optimize_sharpe` y se eligen el lookback y el target
   vol que maximizan el Sharpe *de ese pasado*;
2. con esos parámetros se mide el **año siguiente**, que el optimizador **nunca vio**.

Se avanza un año y se repite hasta el final. Es OOS **de verdad** en la selección de parámetros, no solo en la
señal (que ya es causal por el `shift(1)`). La tabla incluye el lookback/target_vol elegido por ventana: ver
cómo **migra** es parte del análisis.

> **Por qué NO un corte único IS/OOS.** Una alternativa común es partir la muestra en dos (ej. 60% in-sample /
> 40% out-of-sample) y comparar el Sharpe. La descartamos a propósito: el resultado **depende fuertísimo de
> dónde cortes** — calibrando con poca historia temprana el Sharpe OOS da ~0.3, y con más historia ~1.0. Un
> solo corte es arbitrario y puede dar una foto optimista o pesimista según el punto elegido. El walk-forward,
> al recorrer **todos** los períodos re-optimizando, evita esa arbitrariedad (es el análogo temporal del
> barrido de lookback de la sección A).

**Lectura (corrida actual):**

- **Sharpe OOS medio ~0.96 · mediana ~0.70 · 7/9 ventanas positivas (78%)**. El único año netamente malo es
  **2022** (consistente con el stress test). Que la *mayoría* de los años dé positivo —sin que uno solo salve
  el promedio— es buena señal.
- **Drift del óptimo** — el lookback elegido salta entre **6, 18 y 24 meses** según la ventana, y el target vol
  entre 5% y 20%. **El óptimo es inestable**: no hay un parámetro "verdadero", y las ventanas más flojas suelen
  ser las que calibraron con poca historia (eligen un lookback corto que no generaliza). Esto es clave para la
  defensa: en vez de esconderlo, el test lo muestra — aun re-eligiendo parámetros con info parcial y cambiante,
  la mayoría de los años out-of-sample son positivos.

In [ ]:
# Out-of-sample: walk-forward RE-OPTIMIZADO por ventana (recorre toda la linea de tiempo)
from src.robustness.out_of_sample import walk_forward

wf = walk_forward(returns, config)     # re-optimiza (lookback, target_vol) con datos hasta train_end (~5s)
ax = wf.set_index('test_start')['sharpe'].plot(
        kind='bar', figsize=(9, 2.8),
        color=['green' if v >= 0 else 'crimson' for v in wf['sharpe']])
ax.axhline(0, color='gray', lw=0.8)
ax.set_title('Walk-forward — Sharpe out-of-sample por ventana anual (params re-optimizados)')
ax.set_xlabel('inicio del test'); ax.set_ylabel('Sharpe')
plt.tight_layout(); plt.show()

print("Walk-forward (params re-optimizados con datos hasta train_end; test = año siguiente):")
display(wf.round(2))

# --- Lectura automática ---
print("LECTURA — Out-of-sample (walk-forward, todos los periodos)")
print(f"  Sharpe OOS: medio {wf['sharpe'].mean():.2f}, mediana {wf['sharpe'].median():.2f}, "
      f"{(wf['sharpe']>0).sum()}/{len(wf)} ventanas positivas ({(wf['sharpe']>0).mean()*100:.0f}%)")
print(f"  Drift del optimo: lookback {sorted(wf['lookback'].unique())}m, "
      f"target vol {sorted(set((wf['target_vol']*100).round().astype(int)))}% -> "
      f"{'estable' if wf['lookback'].nunique()<=2 else 'INESTABLE (no hay parametro unico)'}")

### Checklist de la sección Robustness
- [ ] Entiendo la **sensibilidad**: re-correr cambiando un parámetro y ver que el Sharpe degrada *suave* y queda positivo (no "filo de cuchillo").
- [ ] Sé por qué el **target vol casi no mueve el Sharpe** (lo domina el vol-scaling del portafolio al 7%) y hasta qué **costo** aguanta (~10x lo real).
- [ ] Entiendo el **stress test**: COVID 2020 (defendió) y 2022 (el peor año), y que contra **buy & hold** la estrategia perdió mucho menos (−5.6% vs −18% del S&P en 2022).
- [ ] Entiendo que el **walk-forward re-optimiza los parámetros por ventana** (OOS real sobre todos los períodos) y por qué descartamos el **corte único IS/OOS** (arbitrario: depende de dónde cortes).
- [ ] Vi que el **óptimo es inestable** entre ventanas (drift) y que aun así la **mayoría de las ventanas OOS son positivas** — evidencia de que no hay sobreajuste grosero.


---
<a id='s7'></a>
## 7. Broker IBKR y trading en vivo — `src/broker/`

Hasta acá todo fue *simulación*. El módulo `broker/` conecta la estrategia a una cuenta **real (o paper)** de
Interactive Brokers vía `ib_insync`, y agrega la maquinaria para **operar de verdad, mes a mes, de forma segura**.
Punto clave: **la ejecución usa exactamente los mismos pesos que el backtest** (`build_scaled_weights`), así que
*lo que se backtestea es lo que se opera*.

### Arquitectura de comunicación
```
TWS / IB Gateway (corriendo en tu PC)
      ↑↓  TCP socket  (7497 = paper · 7496 = real · 4001 = gateway paper)
ib_insync  (cliente Python del protocolo IBKR)
      ↑↓
src/broker/
  ├─ ibkr.py          IBKRConnection: conecta, lee cuenta / posiciones / precios
  ├─ session.py       context manager con un client_id por propósito (evita choques en TWS)
  ├─ orders.py        execute_rebalance() y close_all_positions() (kill switch) + logs
  ├─ runner.py        el "tick": separa CHEQUEAR de EJECUTAR el rebalanceo
  ├─ state.py         estado persistente (el "cerebro" que sobrevive a cerrar el dashboard)
  ├─ monitor.py       portafolio en vivo + P&L no realizado
  └─ market_hours.py  estado del mercado US (regular / pre / after / cerrado)
```

### El ciclo de vida de la estrategia (máquina de estados)
El dashboard es solo una **ventana**: las posiciones reales viven en IBKR y la *intención* de la estrategia vive
en `logs/strategy_state.json` (vía `state.py`). Así, al reconectar al día siguiente, el sistema sabe en qué
estado quedó y puede reconciliar (*catch-up*).
```
   DETENIDA ──start-strategy──▶ ACTIVA ◀──resume── PAUSADA
      ▲                          │  │                 ▲
      │                       halt  └───── resume ────┘
      └──── close-all (kill switch) ◀──┘
```
- **`stopped`** — no enrolada (estado inicial / después del kill switch).
- **`running`** — enrolada y operando: los rebalanceos mensuales están habilitados.
- **`paused`** — enrolada pero **no opera**; mantiene posiciones.

### Rebalanceo mensual: chequear ≠ ejecutar (`runner.py`)
`run_tick` separa dos cosas para que el "chequeo" pueda correr cada pocos segundos (gratis, idempotente) pero la
**ejecución** ocurra **una vez por mes**, cuando cambia el mes calendario:
- **`check_rebalance()`** — ¿toca rebalancear? (la estrategia está `running` y todavía no se rebalanceó este mes). No envía nada.
- **`run_tick()`** — si toca (o si se fuerza con `--force`), calcula los pesos objetivo y ejecuta el rebalanceo.

### Ejecución y seguridad (`orders.py`)
`execute_rebalance()` recibe los `target_weights` de la estrategia y el capital disponible, y:
1. Lee las posiciones actuales en IBKR.
2. Calcula la diferencia en valor `Δ = w_target × capital − posición_actual` y la pasa a acciones.
3. Descarta trades chicos (< 1 acción) para no fragmentar.
4. Envía las órdenes y registra **dos niveles** de log: `logs/trades.csv` (una fila por orden, con precio de fill) y `logs/rebalances.csv` (una fila por evento de rebalanceo).

**Salvaguardas:** `execute_rebalance()` se **niega a operar** si la estrategia está pausada/halted;
`close_all_positions()` es el **kill switch** (cierra todo a mercado). `--dry-run` muestra las órdenes sin
enviarlas. `market_hours.py` avisa si el mercado está cerrado (las market orders en after-hours son poco confiables).

> **En el dashboard:** esto alimenta las páginas **04 · Live Portfolio** (conexión, posiciones, P&L, drift, kill
> switch), **05 · Trade Log** (`logs/trades.csv`) y **06 · Rebalances** (`logs/rebalances.csv`).

### Setup para conectar a TWS (paper trading)
```
1. TWS → Edit → Global Configuration → API → Settings
2. ✅ Enable ActiveX and Socket Clients
3. Socket port: 7497 (paper)   — NO usar 7496 (real) para el TP
4. Agregar 127.0.0.1 a Trusted IPs  → OK  (reiniciar TWS si lo pide)
5. python main.py test-connection      # verifica el socket
6. python main.py monitor              # portafolio en vivo
```


In [ ]:
# Demo de la estructura del módulo broker (sin conectar a TWS)
# Para ver el código completo, abrir src/broker/ibkr.py

import inspect
from src.broker.ibkr import IBKRConnection
from src.broker.orders import execute_rebalance
from src.broker.monitor import print_portfolio_table

print('── IBKRConnection — métodos públicos ──')
methods = [m for m in dir(IBKRConnection) if not m.startswith('_')]
for m in methods:
    sig = inspect.signature(getattr(IBKRConnection, m))
    print(f'  .{m}{sig}')

print('\n── execute_rebalance — firma ──')
print(f'  {inspect.signature(execute_rebalance)}')

print('\n── print_portfolio_table — firma ──')
print(f'  {inspect.signature(print_portfolio_table)}')

In [ ]:
# Demo de dry-run: qué órdenes generaría el rebalanceo actual
# (sin conectar a IBKR — usa capital simulado de $1M)

from src.strategy.portfolio import build_portfolio

target_weights, _ = build_portfolio(returns, config,
                       use_signal_strength=(config['strategy']['signal_mode'] == 'strength'))
latest_weights = target_weights.iloc[-1].dropna()
latest_weights = latest_weights[latest_weights != 0]

capital = 1_000_000
prices_sim = {t: 100.0 for t in latest_weights.index}  # precio simulado $100

print(f'Mes: {target_weights.index[-1].strftime("%Y-%m")} | Capital: ${capital:,.0f}')
print(f'{"─"*60}')
print(f'  {"Ticker":<8} {"Peso":>8} {"Valor ($)":>12} {"Acciones":>10} {"Acción"}')
print(f'{"─"*60}')

for ticker, w in latest_weights.sort_values().items():
    valor = w * capital
    qty = abs(valor) / prices_sim.get(ticker, 100.0)
    action = 'BUY' if w > 0 else 'SELL'
    print(f'  {ticker:<8} {w:>7.2%} {valor:>12,.0f} {qty:>10.0f}  {action}')

print(f'{"─"*60}')
print(f'  Total posiciones: {len(latest_weights)} | '
      f'Long: {(latest_weights>0).sum()} | Short: {(latest_weights<0).sum()}')

---
<a id='s8'></a>
## 8. CLI — `main.py`

La interfaz de línea de comandos está hecha con **Click**. `main.py` **solo orquesta**: lee `config.yaml`, aplica
los overrides de la línea de comandos encima y llama a `src/`. Toda la lógica vive en los módulos.

### Comandos (agrupados por uso)
```
main.py  [--config config.yaml]
│
├─ DATOS
│   ├─ update-data        descarga precios nuevos vía yfinance (incremental)
│   └─ data-status        chequea que universe.py y el parquet sigan alineados
│
├─ INVESTIGACIÓN
│   ├─ backtest           corre el motor    [--lookback --target-vol --start --end --signal]
│   ├─ robustness         tests de robustez [--test all|sensitivity|stress|oos]
│   └─ dashboard          lanza Streamlit   [--port 8501]
│
├─ CONEXIÓN IBKR
│   ├─ test-connection    abre el socket con TWS y lee la cuenta
│   ├─ validate-ibkr      valida que cada ticker sea operable en IBKR  [--out csv]
│   └─ monitor            portafolio en vivo + P&L, refrescando        [--interval 30]
│
└─ TRADING EN VIVO (ciclo de vida)
    ├─ start-strategy     enrola la estrategia (estado → running)
    ├─ strategy-status    muestra el estado persistente (state.py)
    ├─ tick               chequea / ejecuta el rebalanceo mensual  [--force --dry-run --capital]
    ├─ execute            rebalanceo puntual                       [--dry-run --capital]
    ├─ halt / resume      pausa / reanuda la operación (mantiene posiciones)
    └─ close-all          KILL SWITCH: cierra todo a mercado       [--execute --halt/--no-halt]
```

### Patrón: overrides sobre `config.yaml`
Ningún comando hardcodea valores: leen `config.yaml` y aplican los flags del CLI encima. Así
`python main.py backtest` usa el config, y `python main.py backtest --lookback 6 --signal strength` cambia solo eso.


In [ ]:
# Ver el help del CLI
import subprocess, sys
result = subprocess.run([sys.executable, 'main.py', '--help'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Help del comando backtest
result = subprocess.run([sys.executable, 'main.py', 'backtest', '--help'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# Correr el backtest desde el CLI (demostración real)
result = subprocess.run(
    [sys.executable, 'main.py', 'backtest',
     '--lookback', '12', '--signal', 'binary'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

In [ ]:
# Dry-run del execute: ver qué órdenes se generarían
result = subprocess.run(
    [sys.executable, 'main.py', 'execute', '--dry-run'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

---
<a id='s9'></a>
## 9. Dashboard — `src/dashboard/app.py`

Construido con **Streamlit**. Es una **capa de presentación**: no tiene lógica propia, llama a `src/` (las mismas
funciones de este notebook). Al abrir, **(a)** refresca el parquet con yfinance 1× por día y **(b)** corre
`optimize_sharpe` para fijar los sliders (lookback, target-vol) al **óptimo in-sample** de la señal/ponderación
elegidas — el mismo óptimo que reproduce el setup de este notebook.

**Lanzarlo:**
```bash
python main.py dashboard          # o:  streamlit run src/dashboard/app.py
```

### Las 6 páginas y dónde está cada una en este notebook
| Página del dashboard | Qué muestra | Backend (`src/`) | Sección de este notebook |
|---|---|---|---|
| **01 · Overview** | señal por activo, exposición por clase, NAV, KPIs | `strategy/` | §4 (Estrategia) |
| **02 · Backtest** | curva neta vs bruta, drawdown, heatmap, métricas | `backtest/` | §5 (Backtest) |
| **03 · Robustness** | sensibilidad · stress test · walk-forward OOS | `robustness/` | §6 (Robustez) |
| **04 · Live Portfolio** | conexión IBKR, posiciones, P&L, drift, kill switch | `broker/` | §7 (Broker) |
| **05 · Trade Log** | `logs/trades.csv` (una fila por orden) | `broker/orders` | §7 (Broker) |
| **06 · Rebalances** | `logs/rebalances.csv` (una fila por rebalanceo) | `broker/orders` | §7 (Broker) |

### Decisiones técnicas
| Decisión | Razón |
|---|---|
| `@st.cache_data` keyed al **último mes completo** | la estrategia es mensual: el óptimo/backtest se recalculan solo cuando cierra un mes, no en cada rerun |
| sliders como **overrides** del config | explorar variantes sin tocar archivos |
| **Plotly** en vez de matplotlib | gráficos interactivos (zoom, hover, descarga) |
| Live Portfolio con botón "Conectar" | la conexión IBKR es costosa: no conectar al cargar |
| `optimize_sharpe` al abrir | arrancar en el óptimo in-sample (reproducido en el setup de este notebook) |


In [ ]:
# Verificar que el dashboard es importable sin errores y contar sus páginas
import ast, pathlib, re

code_app = pathlib.Path('src/dashboard/app.py').read_text(encoding='utf-8')
try:
    ast.parse(code_app)
    print('✓ src/dashboard/app.py — sintaxis válida')
    print(f'  Líneas de código: {len(code_app.splitlines())}')
    pages = re.findall(r'page\.startswith\(["\'](\d+)["\']\)', code_app)
    print(f'  Páginas despachadas: {len(pages)}  ->  {pages}')
    radio = re.search(r'st\.radio\(\s*"Navegación",\s*(\[[^\]]*\])', code_app, re.S)
    if radio:
        for n in re.findall(r'"([^"]+)"', radio.group(1)):
            print(f'    • {n}')
except SyntaxError as e:
    print(f'✗ Error de sintaxis: {e}')


In [ ]:
# Simular la Página 1 del dashboard (Overview) en el notebook
# (misma lógica que app.py pero con matplotlib en lugar de Streamlit)

from src.strategy.signals import compute_tsmom_signal

signal_latest = compute_tsmom_signal(returns, lookback=config['strategy']['lookback_months']).iloc[-1].dropna()
date_label = returns.index[-1].strftime('%Y-%m')

sig_df = pd.DataFrame({
    'ticker':     signal_latest.index,
    'signal':     signal_latest.values.astype(int),
    'asset_class': [ETF_UNIVERSE.get(t, 'other') for t in signal_latest.index]
})

# Heatmap de señales por asset class
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, ac in zip(axes, ASSET_CLASSES):
    ac_df = sig_df[sig_df['asset_class'] == ac]
    colors_s = ['#4caf50' if v == 1 else '#f44336' for v in ac_df['signal']]
    ax.barh(ac_df['ticker'], ac_df['signal'], color=colors_s, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'{ac.capitalize()} ({len(ac_df)} ETFs)')
    ax.set_xlim(-1.5, 1.5)
    ax.set_xticks([-1, 0, 1])
    ax.set_xticklabels(['SHORT', '', 'LONG'])

plt.suptitle(f'Señales TSMOM — {date_label} (verde=LONG, rojo=SHORT)', fontsize=12)
plt.tight_layout(); plt.show()

n_long  = (signal_latest == 1).sum()
n_short = (signal_latest == -1).sum()
print(f'Posiciones LONG: {n_long} | SHORT: {n_short} | Net: {n_long - n_short:+d}')

---
<a id='s10'></a>
## 10. Configuración central — `config.yaml`

**Toda la parametrización vive en un único archivo.** Cambiar un valor acá cambia el comportamiento de todo el
sistema sin tocar código. (La celda de abajo imprime el `config.yaml` real.)

```yaml
strategy:
  lookback_months: 12        # ventana TSMOM base (el dashboard la re-optimiza al abrir)
  target_volatility: 0.10    # riesgo objetivo POR ACTIVO (vol-scaling)
  rebalancing: monthly
  max_position_weight: 0.15  # tope de peso por activo (±15%)
  vol_com_months: 3          # EWMA center-of-mass ≈ 60 ruedas
  signal_mode: strength      # binary (Moskowitz) | strength (Baz)   [default: strength]
  lookbacks: [3, 6, 12]      # horizontes para multihorizon (fuera del dashboard)
  weighting: pooled          # pooled (1/N global) | class_balanced (presupuesto por clase)
  use_crash_filter: true     # Daniel & Moskowitz (2016): recorta señal en pánico
  use_portfolio_vol_scaling: true   # Moreira & Muir (2017): escala la cartera entera
  target_portfolio_vol: 0.07 # vol objetivo del PORTAFOLIO (no por activo)

transaction_costs:
  bid_ask_spread: 0.001      # 10 bps — ETFs líquidos
  commission_pct: 0.0005     # 5 bps — IBKR

backtest:
  start_date: "2012-01-01"
  end_date: null             # null = ventana ABIERTA (hasta el último mes completo, auto-avanza)
  initial_capital: 1_000_000

data:
  path: "Data/etf_prices.parquet"
  yfinance_update: false
  min_history_months: 24     # piso de historia por ETF para entrar al backtest

broker:
  host: "127.0.0.1"
  port: 7497                 # 7497 paper · 7496 real · 4001 gateway paper
  client_id: 1
```

> Dos capas que el TSMOM clásico no tiene: **`use_crash_filter`** (Daniel-Moskowitz 2016) y
> **`use_portfolio_vol_scaling`** con `target_portfolio_vol` (Moreira-Muir 2017). Se justifican en §4.4 y §6.


In [ ]:
# Mostrar el config actual
import yaml
with open('config.yaml') as f:
    config_loaded = yaml.safe_load(f)

print('config.yaml actual:')
print('─' * 40)
for section, values in config_loaded.items():
    print(f'[{section}]')
    if isinstance(values, dict):
        for k, v in values.items():
            print(f'  {k}: {v}')
    else:
        print(f'  {values}')
    print()

---
<a id='s11'></a>
## 11. Demo End-to-End

Esta sección demuestra el sistema completo en acción:
1. Cargar datos → 2. Correr backtest → 3. Robustez → 4. Generar órdenes → 5. Comparar configuraciones

In [ ]:
# ── 1. Pipeline completo de punta a punta — la configuración OPERADA ──
# (señal del config = 'strength', en el óptimo fijado por el setup)
from src.backtest.metrics import print_report

res = BacktestEngine(config).run(
    returns, use_signal_strength=(config['strategy']['signal_mode'] == 'strength'))

s = config['strategy']
print_report(res.metrics,
             title=f"TSMOM — {s['signal_mode']} · lookback {s['lookback_months']}m · target-vol {s['target_volatility']:.0%}")


In [ ]:
# ── 2. Comparación: señal binaria vs señal continua ──
from copy import deepcopy
from src.backtest.metrics import sharpe_ratio, annualized_return, max_drawdown

results_bin  = BacktestEngine(config).run(returns, use_signal_strength=False)
results_cont = BacktestEngine(config).run(returns, use_signal_strength=True)

comparison = pd.DataFrame({
    'Señal Binaria (Moskowitz)': [
        sharpe_ratio(results_bin.portfolio_returns),
        annualized_return(results_bin.portfolio_returns),
        max_drawdown(results_bin.portfolio_returns),
    ],
    'Señal Continua (Baz)': [
        sharpe_ratio(results_cont.portfolio_returns),
        annualized_return(results_cont.portfolio_returns),
        max_drawdown(results_cont.portfolio_returns),
    ],
}, index=['Sharpe Ratio', 'Ann. Return', 'Max Drawdown'])

print('Comparación de señales:')
print(comparison.applymap(lambda x: f'{x:.2f}' if abs(x) > 1 else f'{x:.2%}'))

In [ ]:
# ── 3. Equity curves comparadas ──
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(results_bin.cumulative.index, results_bin.cumulative.values,
        label='Binaria (Moskowitz 2012)', color='navy', linewidth=1.8)
ax.plot(results_cont.cumulative.index, results_cont.cumulative.values,
        label='Continua (Baz et al. 2015)', color='darkorange', linewidth=1.8, linestyle='--')

ax.set_yscale('log')
ax.set_title('Comparación de Señales — Binaria vs Continua', fontsize=13)
ax.set_ylabel('Valor del portafolio (log, base 1)')
ax.legend()

plt.tight_layout(); plt.show()

In [ ]:
# ── 4. Resumen de la estructura de archivos del proyecto ──
import pathlib

root = pathlib.Path('.')
print('Estructura del proyecto:')
for path in sorted(root.rglob('*.py')):
    parts = path.parts
    if any(p in parts for p in ['.git', '__pycache__', '.ipynb_checkpoints']):
        continue
    indent = '  ' * (len(parts) - 1)
    lines = len(path.read_text(encoding='utf-8', errors='ignore').splitlines())
    print(f'{indent}{path.name:35} ({lines:4d} líneas)')

print()
# Contar también archivos de config y datos
for fname in ['config.yaml', 'requirements.txt', 'Data/etf_prices.parquet']:
    p = pathlib.Path(fname)
    if p.exists():
        size = p.stat().st_size
        print(f'  {fname:35} ({size/1024:.1f} KB)')

---
<a id='s12'></a>
## 12. Resumen de decisiones de diseño

| Decisión | Alternativa considerada | Por qué lo elegimos |
|---|---|---|
| **ETFs** en vez de futuros | futuros (como el paper original) | datos gratis (yfinance) y ejecutables en IBKR paper sin margen de futuros |
| Universo **curado 205 → 133** (corr ≥ 0.95) | usar los 205 crudos | 1 instrumento líquido por mercado (Moskowitz 2012): saca duplicados, no diversidad |
| Señal **`strength`** por default | `binary` / `multihorizon` | mejor Sharpe al óptimo y más estable fuera de muestra; `binary` queda como opción robusta; `multihorizon` rinde peor en ETF mensual |
| **`pooled`** (1/N global) por default | `class_balanced` | en este universo `class_balanced` no mejora el Sharpe (las clases no-equity tienen menos señal limpia); se deja como opción para *justificar* la elección |
| **Filtro de crash** (Daniel-Moskowitz 2016) | sin filtro | recorta exposición en pánico → evita los *momentum crashes* (rebotes violentos) |
| **Vol-scaling del portafolio** (Moreira-Muir 2017) | solo vol-scaling por activo | apunta la cartera entera a 7% de vol → estabiliza el riesgo y aplana la sensibilidad al target por activo |
| Sharpe sobre **exceso de T-bill 1m** | rf = 0 (como la literatura de futuros) | operamos ETF cash apalancados: corresponde restar financiamiento (Sharpe 1994) |
| **Rebalanceo mensual** | semanal / diario | menor turnover (≈4×) y costos; consistente con el paper |
| **Ventana abierta** (`end_date: null`) | ventana fija | avanza sola a medida que cierran los meses; nunca entra un mes parcial |
| **`config.yaml`** centralizado | parámetros en cada módulo | una sola fuente de verdad → reproducibilidad |
| **`.shift(1)`** en vol y señal | calcular en `t` | cero look-ahead bias |
| El backtest **es** lo que se opera | motores separados | `BacktestEngine` y la ejecución IBKR comparten `build_scaled_weights` |

## Cómo correr el sistema
```bash
pip install -r requirements.txt          # dependencias

python main.py update-data               # actualizar precios (yfinance, incremental)
python main.py data-status               # chequear universo vs parquet

python main.py backtest                  # backtest con el config
python main.py backtest --lookback 9 --target-vol 0.12 --signal strength
python main.py robustness --test all     # sensibilidad + stress + out-of-sample
python main.py dashboard                  # dashboard interactivo (6 páginas)

python main.py test-connection           # probar el socket con TWS
python main.py execute --dry-run         # ver órdenes del próximo rebalanceo
python main.py start-strategy            # enrolar la estrategia en vivo
python main.py tick --dry-run            # ¿toca rebalancear este mes?
python main.py monitor                   # portafolio en vivo + P&L
python main.py close-all                 # kill switch (dry-run; agregar --execute para real)
```

---
*Notebook único de documentación del TP — Trend Following / Time Series Momentum · Ingeniería Financiera (F414) · UdeSA.*
